# 出租车 GPS 全流程

单文件完整流程：数据预处理、研究区提取、拥堵识别、OD 提取、道路匹配、最优路径和求交分析。默认 `EXECUTE_PIPELINE=False`，填写真实路径并确认参数后再开启执行。


## 0. 公共配置与状态

集中填写 7 天数据、空间边界、两套路网、输出目录和算法参数。


In [ ]:
from __future__ import annotations

import csv
import hashlib
import heapq
import importlib
import json
import math
import random
import tempfile
import time
import traceback
from contextlib import ExitStack
from dataclasses import asdict, dataclass, field
from datetime import date, datetime, time as clock_time, timedelta, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Callable, Iterable, Iterator, Mapping, Sequence

EXECUTE_PIPELINE = False


def _optional_import(module_name: str) -> tuple[Any | None, str | None]:
    try:
        return importlib.import_module(module_name), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


np, _np_error = _optional_import("numpy")
pd, _pd_error = _optional_import("pandas")
gpd, _gpd_error = _optional_import("geopandas")
rasterio, _rasterio_top_error = _optional_import("rasterio")
rasterio_features, _rasterio_features_error = _optional_import("rasterio.features")
rasterio_transform, _rasterio_transform_error = _optional_import("rasterio.transform")
_rasterio_error = (
    _rasterio_top_error or _rasterio_features_error or _rasterio_transform_error
)
scipy_signal, _scipy_error = _optional_import("scipy.signal")
shapely_geometry, _shapely_geometry_error = _optional_import("shapely.geometry")
shapely_ops, _shapely_ops_error = _optional_import("shapely.ops")
shapely_strtree, _shapely_strtree_error = _optional_import("shapely.strtree")
shapely_top, _shapely_top_error = _optional_import("shapely")
pyproj, _pyproj_error = _optional_import("pyproj")
nx, _nx_error = _optional_import("networkx")
pyogrio, _pyogrio_error = _optional_import("pyogrio")

OPTIONAL_IMPORT_ERRORS = {
    "numpy": _np_error,
    "pandas": _pd_error,
    "geopandas": _gpd_error,
    "rasterio": _rasterio_error,
    "scipy": _scipy_error,
    "shapely": _shapely_geometry_error or _shapely_ops_error or _shapely_strtree_error,
    "pyproj": _pyproj_error,
    "networkx": _nx_error,
    "pyogrio": _pyogrio_error,
}


def _default_run_id() -> str:
    return datetime.now(timezone.utc).strftime("taxi_gps_%Y%m%dT%H%M%SZ")




**关键配置：** `road_classification_path` 用于道路等级匹配，`routing_road_path` 用于最优路径，两者不能互换。


In [ ]:
@dataclass(frozen=True)
class PipelineConfig:
    # 执行与输入。日期键写作 YYYY-MM-DD；默认没有任何数据路径。
    execute_pipeline: bool = EXECUTE_PIPELINE
    input_tsv_by_date: Mapping[str, Path] = field(default_factory=dict)
    administrative_boundary_path: Path | None = None
    district_boundary_path: Path | None = None
    final_study_area_path: Path | None = None
    road_classification_path: Path | None = None
    routing_road_path: Path | None = None

    # 输出。运行目录只在 execute_pipeline=True 后显式创建。
    output_root: Path = Path("runs")
    run_id: str = field(default_factory=_default_run_id)

    # 固定研究期与坐标系。
    dates: tuple[str, ...] = (
        "2017-03-01", "2017-03-02", "2017-03-03", "2017-03-04",
        "2017-03-05", "2017-03-06", "2017-03-07",
    )
    timezone_name: str = "Asia/Shanghai"
    geographic_crs: str = "EPSG:4326"
    projected_crs: str = "EPSG:32650"

    # 预处理与轨迹特征。
    external_sort_chunk_rows: int = 500_000
    csv_chunk_rows: int = 2_000_000
    max_gap_sec: int = 1_800
    drift_speed_kmh: float = 80.0
    drift_distance_m: float = 1_000.0
    drift_angle_deg: float = 30.0
    china_lon_bounds: tuple[float, float] = (73.0, 135.0)
    china_lat_bounds: tuple[float, float] = (18.0, 53.0)

    # KDE 与候选研究区。
    kde_cell_m: float = 150.0
    kde_radius_m: float = 800.0
    daily_hotspot_top_fraction: float = 0.03
    persistence_days: int = 7
    min_component_cells: int = 3

    # 停车、拥堵与 OD。
    parking_zero_speed_kmh: float = 0.0
    parking_min_duration_sec: int = 3_600
    congestion_speed_kmh: float = 20.0
    congestion_min_duration_sec: int = 240
    speed_column: str = "speed_gps_kmh"

    # 两套独立路网参数。
    road_class_field: str = "type"
    road_match_tolerance_m: float = 30.0
    routing_required_fields: tuple[str, ...] = (
        "osm_id", "fclass", "oneway", "layer", "bridge", "tunnel", "geometry",
    )
    routing_snap_tolerance_m: float = 500.0
    alternative_max_ratio: float = 1.20
    alternative_max_attempts: int = 5
    max_paths_per_trip: int = 2
    random_seed: int = 42

    # 时间与绕行。
    morning_start: str = "07:00"
    morning_end: str = "09:00"
    evening_start: str = "17:00"
    evening_end: str = "19:00"
    detour_ratio_threshold: float = 1.5

    @property
    def run_dir(self) -> Path:
        return self.output_root / self.run_id


class StageStatus(str, Enum):
    SUCCESS = "SUCCESS"
    SKIPPED = "SKIPPED"
    FAILED = "FAILED"


@dataclass
class StageResult:
    stage: str
    status: StageStatus
    message: str
    inputs: list[str] = field(default_factory=list)
    outputs: list[str] = field(default_factory=list)
    elapsed_sec: float = 0.0
    error: str | None = None
    metrics: dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> dict[str, Any]:
        payload = asdict(self)
        payload["status"] = self.status.value
        return payload




In [ ]:
def stage_skipped(
    stage: str,
    message: str,
    *,
    inputs: Iterable[Path | str] = (),
    outputs: Iterable[Path | str] = (),
) -> StageResult:
    return StageResult(
        stage=stage,
        status=StageStatus.SKIPPED,
        message=message,
        inputs=[str(x) for x in inputs],
        outputs=[str(x) for x in outputs],
    )


def stage_failed(
    stage: str,
    message: str,
    exc: BaseException,
    *,
    inputs: Iterable[Path | str] = (),
    outputs: Iterable[Path | str] = (),
    elapsed_sec: float = 0.0,
) -> StageResult:
    return StageResult(
        stage=stage,
        status=StageStatus.FAILED,
        message=message,
        inputs=[str(x) for x in inputs],
        outputs=[str(x) for x in outputs],
        elapsed_sec=elapsed_sec,
        error="".join(traceback.format_exception_only(type(exc), exc)).strip(),
    )


def require_modules(*names: str) -> None:
    missing = [name for name in names if OPTIONAL_IMPORT_ERRORS.get(name)]
    if missing:
        details = "; ".join(f"{name}: {OPTIONAL_IMPORT_ERRORS[name]}" for name in missing)
        raise RuntimeError(f"缺少可选依赖：{details}")


def require_columns(frame: Any, columns: Iterable[str], label: str) -> None:
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise ValueError(f"{label} 缺少字段 {missing}；实际字段为 {list(frame.columns)}")


def require_crs(frame: Any, label: str) -> None:
    if getattr(frame, "crs", None) is None:
        raise ValueError(f"{label} 缺少 CRS，不能安全执行空间计算")


def ensure_new_run_directory(config: PipelineConfig) -> Path:
    if not config.execute_pipeline:
        raise RuntimeError("执行开关关闭时禁止创建运行目录")
    run_dir = config.run_dir
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def dependency_report() -> list[dict[str, str]]:
    report = []
    for name, error in OPTIONAL_IMPORT_ERRORS.items():
        report.append({
            "dependency": name,
            "status": "AVAILABLE" if error is None else "MISSING",
            "detail": "" if error is None else error,
        })
    return report


config = PipelineConfig()


## 1. 数据预处理

将原始 GPS 转成按车辆和时间排序、包含速度与方向特征的每日数据。


In [ ]:
from __future__ import annotations

SOURCE_COLUMN_COUNT = 13
CORE_COLUMNS = [
    "group_id", "taxi_id", "timestamp", "speed", "latitude", "longitude",
    "direction", "positioning_valid", "occupied",
]
FEATURE_COLUMNS = [
    "group_id", "taxi_id", "timestamp", "latitude", "longitude", "direction",
    "occupied", "positioning_valid", "dt_sec", "speed_gps_kmh",
    "speed_reported_kmh", "heading_delta_deg", "trip_break", "bad_coord",
    "out_of_day", "event_type", "is_outlier_drift",
]


def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    radius_m = 6_371_008.8
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = (
        math.sin(dphi / 2.0) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2.0) ** 2
    )
    return 2.0 * radius_m * math.asin(math.sqrt(min(1.0, max(0.0, a))))


def circular_heading_diff(first_deg: float, second_deg: float) -> float:
    return abs((float(second_deg) - float(first_deg) + 180.0) % 360.0 - 180.0)


def beijing_day_epoch_bounds(day_text: str) -> tuple[int, int]:
    require_modules("pandas")
    start = pd.Timestamp(day_text, tz="Asia/Shanghai")
    end = start + pd.Timedelta(days=1)
    return int(start.timestamp()), int(end.timestamp())


def transform_raw_row(row: Sequence[str]) -> list[str]:
    if len(row) != SOURCE_COLUMN_COUNT:
        raise ValueError(f"原始行应有 {SOURCE_COLUMN_COUNT} 列，实际 {len(row)} 列")
    status_text = row[12]
    timestamp = int(row[2])
    latitude = float(row[4]) / 100_000.0
    longitude = float(row[5]) / 100_000.0
    return [
        row[0],
        row[1],
        str(timestamp),
        row[3],
        str(latitude),
        str(longitude),
        row[6],
        str(int("定位有效" in status_text)),
        str(int("重车" in status_text)),
    ]


def _core_sort_key(row: Sequence[str]) -> tuple[str, int]:
    return str(row[1]), int(row[2])


def _write_sorted_chunk(
    rows: list[list[str]],
    directory: Path,
    chunk_number: int,
) -> Path:
    rows.sort(key=_core_sort_key)
    chunk_path = directory / f"chunk_{chunk_number:06d}.csv"
    with chunk_path.open("x", encoding="utf-8", newline="") as handle:
        csv.writer(handle).writerows(rows)
    return chunk_path




**原始数据转换：** 校验 13 列输入，并按 `(taxi_id, timestamp)` 做核外排序。


In [ ]:
def convert_tsv_to_core_csv(
    input_path: Path,
    output_path: Path,
    *,
    chunk_rows: int = 500_000,
    temp_parent: Path | None = None,
) -> dict[str, int]:
    input_path = Path(input_path)
    output_path = Path(output_path)
    if chunk_rows < 1:
        raise ValueError("chunk_rows 必须为正整数")
    if not input_path.is_file():
        raise FileNotFoundError(f"找不到原始 TSV：{input_path}")
    if input_path.resolve() == output_path.resolve():
        raise ValueError("输出不能覆盖输入 TSV")
    if output_path.exists():
        raise FileExistsError(f"输出已存在：{output_path}")

    rows_read = 0
    bad_rows = 0
    chunk_paths: list[Path] = []
    buffer: list[list[str]] = []
    temp_base = Path(temp_parent) if temp_parent is not None else output_path.parent
    temp_base.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory(prefix="taxi_sort_", dir=temp_base) as work:
        work_dir = Path(work)
        with input_path.open("r", encoding="utf-8", newline="") as source:
            for row in csv.reader(source, delimiter="\t"):
                try:
                    buffer.append(transform_raw_row(row))
                    rows_read += 1
                except (TypeError, ValueError, IndexError):
                    bad_rows += 1
                    continue
                if len(buffer) >= chunk_rows:
                    chunk_paths.append(
                        _write_sorted_chunk(buffer, work_dir, len(chunk_paths))
                    )
                    buffer = []
        if buffer:
            chunk_paths.append(_write_sorted_chunk(buffer, work_dir, len(chunk_paths)))

        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("x", encoding="utf-8-sig", newline="") as target:
            writer = csv.writer(target)
            writer.writerow(CORE_COLUMNS)
            with ExitStack() as stack:
                readers = [
                    csv.reader(
                        stack.enter_context(path.open("r", encoding="utf-8", newline=""))
                    )
                    for path in chunk_paths
                ]
                writer.writerows(heapq.merge(*readers, key=_core_sort_key))

    return {
        "rows_written": rows_read,
        "bad_rows_skipped": bad_rows,
        "sort_chunks": len(chunk_paths),
    }




In [ ]:
def is_twoside_drift(
    previous: Mapping[str, float],
    current: Mapping[str, float],
    following: Mapping[str, float],
    *,
    speed_limit_kmh: float,
    distance_limit_m: float,
    angle_limit_deg: float,
) -> bool:
    d_pre = haversine_m(
        current["latitude"], current["longitude"],
        previous["latitude"], previous["longitude"],
    )
    d_next = haversine_m(
        current["latitude"], current["longitude"],
        following["latitude"], following["longitude"],
    )
    d_pre_next = haversine_m(
        previous["latitude"], previous["longitude"],
        following["latitude"], following["longitude"],
    )
    dt_pre = current["timestamp"] - previous["timestamp"]
    dt_next = following["timestamp"] - current["timestamp"]
    dt_pre_next = following["timestamp"] - previous["timestamp"]
    if dt_pre <= 0 or dt_next <= 0 or dt_pre_next <= 0:
        return False

    speed_pre = d_pre / dt_pre * 3.6
    speed_next = d_next / dt_next * 3.6
    speed_pre_next = d_pre_next / dt_pre_next * 3.6
    speed_flag = (
        speed_pre > speed_limit_kmh
        and speed_next > speed_limit_kmh
        and speed_pre_next < speed_limit_kmh
    )
    distance_flag = (
        d_pre > distance_limit_m
        and d_next > distance_limit_m
        and d_pre_next < distance_limit_m
    )
    angle_flag = False
    if d_pre > 0 and d_next > 0:
        cosine = (
            d_pre * d_pre + d_next * d_next - d_pre_next * d_pre_next
        ) / (2.0 * d_pre * d_next)
        angle = math.degrees(math.acos(max(-1.0, min(1.0, cosine))))
        angle_flag = angle < angle_limit_deg
    return bool(speed_flag or distance_flag or angle_flag)




**轨迹特征：** 时间间隔异常时不跨点计算速度、方向或上下客事件。


In [ ]:
def engineer_vehicle_features(
    vehicle_frame: Any,
    day_text: str,
    config: PipelineConfig,
) -> Any:
    require_modules("pandas", "numpy")
    require_columns(vehicle_frame, CORE_COLUMNS, "核心轨迹")
    frame = vehicle_frame.copy()
    frame["timestamp"] = pd.to_numeric(frame["timestamp"], errors="coerce")
    frame = frame.dropna(subset=["timestamp"]).copy()
    frame["timestamp"] = frame["timestamp"].astype("int64")
    frame = (
        frame.sort_values("timestamp", kind="mergesort")
        .drop_duplicates("timestamp", keep="first")
        .reset_index(drop=True)
    )
    numeric_columns = ["latitude", "longitude", "direction", "occupied", "positioning_valid"]
    for column in numeric_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame["speed_reported_kmh"] = pd.to_numeric(frame["speed"], errors="coerce")

    n_rows = len(frame)
    dt_values = np.full(n_rows, np.nan)
    gps_speed = np.full(n_rows, np.nan)
    heading_delta = np.full(n_rows, np.nan)
    trip_break = np.ones(n_rows, dtype="int8")
    event_type = np.full(n_rows, "", dtype=object)
    drift = np.zeros(n_rows, dtype="int8")

    timestamps = frame["timestamp"].to_numpy(dtype="int64")
    latitudes = frame["latitude"].to_numpy(dtype="float64")
    longitudes = frame["longitude"].to_numpy(dtype="float64")
    directions = frame["direction"].to_numpy(dtype="float64")
    occupied = frame["occupied"].fillna(0).astype("int8").to_numpy()

    for index in range(1, n_rows):
        dt = int(timestamps[index] - timestamps[index - 1])
        if 0 < dt <= config.max_gap_sec:
            trip_break[index] = 0
            dt_values[index] = dt
            if np.isfinite(
                [latitudes[index - 1], longitudes[index - 1],
                 latitudes[index], longitudes[index]]
            ).all():
                distance_m = haversine_m(
                    latitudes[index - 1], longitudes[index - 1],
                    latitudes[index], longitudes[index],
                )
                gps_speed[index] = distance_m / dt * 3.6
            if np.isfinite([directions[index - 1], directions[index]]).all():
                heading_delta[index] = circular_heading_diff(
                    directions[index - 1], directions[index]
                )
            if occupied[index - 1] == 0 and occupied[index] == 1:
                event_type[index] = "pickup"
            elif occupied[index - 1] == 1 and occupied[index] == 0:
                event_type[index] = "dropoff"

    for index in range(1, n_rows - 1):
        values = [
            latitudes[index - 1], longitudes[index - 1], timestamps[index - 1],
            latitudes[index], longitudes[index], timestamps[index],
            latitudes[index + 1], longitudes[index + 1], timestamps[index + 1],
        ]
        if not np.isfinite(values).all():
            continue
        previous = {
            "latitude": latitudes[index - 1],
            "longitude": longitudes[index - 1],
            "timestamp": timestamps[index - 1],
        }
        current = {
            "latitude": latitudes[index],
            "longitude": longitudes[index],
            "timestamp": timestamps[index],
        }
        following = {
            "latitude": latitudes[index + 1],
            "longitude": longitudes[index + 1],
            "timestamp": timestamps[index + 1],
        }
        drift[index] = int(
            is_twoside_drift(
                previous, current, following,
                speed_limit_kmh=config.drift_speed_kmh,
                distance_limit_m=config.drift_distance_m,
                angle_limit_deg=config.drift_angle_deg,
            )
        )

    day_start, day_end = beijing_day_epoch_bounds(day_text)
    bad_coord = ~(
        frame["longitude"].between(*config.china_lon_bounds, inclusive="both")
        & frame["latitude"].between(*config.china_lat_bounds, inclusive="both")
    )
    frame["dt_sec"] = dt_values
    frame["speed_gps_kmh"] = np.round(gps_speed, 2)
    frame["heading_delta_deg"] = np.round(heading_delta, 1)
    frame["trip_break"] = trip_break
    frame["bad_coord"] = bad_coord.astype("int8")
    frame["out_of_day"] = (
        (frame["timestamp"] < day_start) | (frame["timestamp"] >= day_end)
    ).astype("int8")
    frame["event_type"] = event_type
    frame["is_outlier_drift"] = drift
    return frame[FEATURE_COLUMNS]




In [ ]:
def _iter_complete_vehicle_blocks(
    csv_path: Path,
    *,
    chunksize: int,
) -> Iterator[Any]:
    require_modules("pandas")
    pending = pd.DataFrame()
    for chunk in pd.read_csv(
        csv_path,
        chunksize=chunksize,
        encoding="utf-8-sig",
        dtype={"taxi_id": str, "group_id": str},
    ):
        data = pd.concat([pending, chunk], ignore_index=True) if len(pending) else chunk
        if data.empty:
            continue
        last_taxi = data.iloc[-1]["taxi_id"]
        ready_mask = data["taxi_id"] != last_taxi
        ready = data.loc[ready_mask]
        pending = data.loc[~ready_mask].copy()
        for _, vehicle in ready.groupby("taxi_id", sort=False):
            yield vehicle
    if len(pending):
        for _, vehicle in pending.groupby("taxi_id", sort=False):
            yield vehicle


def engineer_features_csv(
    core_csv: Path,
    output_csv: Path,
    day_text: str,
    config: PipelineConfig,
) -> dict[str, int]:
    require_modules("pandas", "numpy")
    core_csv = Path(core_csv)
    output_csv = Path(output_csv)
    if not core_csv.is_file():
        raise FileNotFoundError(f"找不到核心 CSV：{core_csv}")
    if output_csv.exists():
        raise FileExistsError(f"输出已存在：{output_csv}")
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    first_write = True
    vehicles = 0
    rows_written = 0
    for vehicle in _iter_complete_vehicle_blocks(
        core_csv, chunksize=config.csv_chunk_rows
    ):
        features = engineer_vehicle_features(vehicle, day_text, config)
        features.to_csv(
            output_csv,
            mode="x" if first_write else "a",
            header=first_write,
            index=False,
            encoding="utf-8",
        )
        first_write = False
        vehicles += 1
        rows_written += len(features)
    if first_write:
        pd.DataFrame(columns=FEATURE_COLUMNS).to_csv(
            output_csv, mode="x", index=False, encoding="utf-8"
        )
    return {"vehicles": vehicles, "rows_written": rows_written}


## 2. 研究区提取

先生成 7 天持续热点候选区，再由 ArcGIS 人工形成最终研究区。


**每日核密度：** 以 150 米像元和 800 米核半径生成 7 幅密度栅格。


In [ ]:
from __future__ import annotations

@dataclass(frozen=True)
class RasterGridSpec:
    xmin: float
    ymin: float
    xmax: float
    ymax: float
    cell_size: float
    width: int
    height: int
    crs: str

    @property
    def transform(self) -> Any:
        require_modules("rasterio")
        return rasterio_transform.from_origin(
            self.xmin, self.ymax, self.cell_size, self.cell_size
        )


def load_union_geometry(path: Path, target_crs: str) -> tuple[Any, Any]:
    require_modules("geopandas", "shapely")
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"找不到空间数据：{path}")
    frame = gpd.read_file(path)
    require_crs(frame, str(path))
    if frame.empty:
        raise ValueError(f"空间数据为空：{path}")
    frame = frame.loc[frame.geometry.notna() & ~frame.geometry.is_empty].copy()
    if frame.empty:
        raise ValueError(f"空间数据没有有效几何：{path}")
    frame = frame.to_crs(target_crs)
    geometry = frame.geometry.union_all() if hasattr(frame.geometry, "union_all") else frame.unary_union
    if geometry is None or geometry.is_empty:
        raise ValueError(f"合并后几何为空：{path}")
    return geometry, frame


def strict_contains_xy(polygon: Any, longitudes: Any, latitudes: Any) -> Any:
    require_modules("numpy", "shapely")
    x_values = np.asarray(longitudes, dtype="float64")
    y_values = np.asarray(latitudes, dtype="float64")
    valid = np.isfinite(x_values) & np.isfinite(y_values)
    result = np.zeros(len(x_values), dtype=bool)
    if not valid.any():
        return result
    if hasattr(shapely_top, "contains_xy"):
        result[valid] = shapely_top.contains_xy(
            polygon, x_values[valid], y_values[valid]
        )
    else:
        point_class = shapely_geometry.Point
        result[valid] = [
            polygon.contains(point_class(float(x), float(y)))
            for x, y in zip(x_values[valid], y_values[valid])
        ]
    return result


def make_kde_grid(
    projected_boundary: Any,
    *,
    cell_size: float,
    crs: str,
) -> RasterGridSpec:
    xmin, ymin, xmax, ymax = projected_boundary.bounds
    xmin = math.floor(xmin / cell_size) * cell_size
    ymin = math.floor(ymin / cell_size) * cell_size
    xmax = math.ceil(xmax / cell_size) * cell_size
    ymax = math.ceil(ymax / cell_size) * cell_size
    width = int(round((xmax - xmin) / cell_size))
    height = int(round((ymax - ymin) / cell_size))
    if width <= 0 or height <= 0:
        raise ValueError("行政边界无法生成有效 KDE 网格")
    return RasterGridSpec(xmin, ymin, xmax, ymax, cell_size, width, height, crs)


def quartic_kernel(radius_m: float, cell_size_m: float) -> Any:
    require_modules("numpy")
    radius_cells = max(1, int(math.ceil(radius_m / cell_size_m)))
    yy, xx = np.mgrid[
        -radius_cells: radius_cells + 1,
        -radius_cells: radius_cells + 1,
    ]
    distance_cells = np.sqrt(
        xx.astype("float64") ** 2 + yy.astype("float64") ** 2
    )
    distance_m = distance_cells * float(cell_size_m)
    kernel = np.zeros_like(distance_m)
    inside = distance_m < float(radius_m)
    kernel[inside] = (1.0 - (distance_m[inside] / radius_m) ** 2) ** 2
    radius_in_cells = float(radius_m) / float(cell_size_m)
    kernel *= 3.0 / (math.pi * radius_in_cells ** 2)
    return kernel




In [ ]:
def _fft_same_convolution(histogram: Any, kernel: Any) -> Any:
    require_modules("numpy")
    target_shape = [
        histogram.shape[index] + kernel.shape[index] - 1 for index in range(2)
    ]

    def next_power_of_two(value: int) -> int:
        return 1 << max(0, value - 1).bit_length()

    fft_shape = tuple(next_power_of_two(value) for value in target_shape)
    histogram_fft = np.fft.rfft2(histogram, fft_shape)
    kernel_fft = np.fft.rfft2(kernel, fft_shape)
    full = np.fft.irfft2(histogram_fft * kernel_fft, fft_shape)
    full = full[: target_shape[0], : target_shape[1]]
    row0 = (kernel.shape[0] - 1) // 2
    col0 = (kernel.shape[1] - 1) // 2
    return full[
        row0: row0 + histogram.shape[0],
        col0: col0 + histogram.shape[1],
    ]


class HistogramAccumulator:
    def __init__(
        self,
        grid: RasterGridSpec,
        *,
        search_radius_m: float,
    ) -> None:
        require_modules("numpy")
        self.grid = grid
        self.search_radius_m = float(search_radius_m)
        self.histogram = np.zeros((grid.height, grid.width), dtype="float64")
        self.n_points_added = 0

    def add(self, x_values: Any, y_values: Any) -> None:
        x_values = np.asarray(x_values, dtype="float64")
        y_values = np.asarray(y_values, dtype="float64")
        valid = (
            np.isfinite(x_values)
            & np.isfinite(y_values)
            & (x_values >= self.grid.xmin)
            & (x_values < self.grid.xmax)
            & (y_values >= self.grid.ymin)
            & (y_values < self.grid.ymax)
        )
        if not valid.any():
            return
        histogram, _, _ = np.histogram2d(
            y_values[valid],
            x_values[valid],
            bins=[self.grid.height, self.grid.width],
            range=[
                [self.grid.ymin, self.grid.ymax],
                [self.grid.xmin, self.grid.xmax],
            ],
        )
        self.histogram += histogram
        self.n_points_added += int(valid.sum())

    def density(self) -> Any:
        kernel = quartic_kernel(self.search_radius_m, self.grid.cell_size)
        smoothed = _fft_same_convolution(self.histogram, kernel)
        density = np.flipud(smoothed)
        peak = float(density.max()) if density.size else 0.0
        noise_floor = peak * 1e-6 if peak > 0 else 0.0
        density[density < noise_floor] = 0.0
        density *= (1_000.0 / self.grid.cell_size) ** 2
        return np.clip(density, 0.0, None).astype("float32")




In [ ]:
def write_single_band_raster(
    array: Any,
    output_path: Path,
    grid: RasterGridSpec,
    *,
    description: str,
    nodata: float = 0.0,
) -> None:
    require_modules("rasterio", "numpy")
    output_path = Path(output_path)
    if output_path.exists():
        raise FileExistsError(f"输出已存在：{output_path}")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        output_path,
        "w",
        driver="GTiff",
        height=array.shape[0],
        width=array.shape[1],
        count=1,
        dtype=array.dtype,
        crs=grid.crs,
        transform=grid.transform,
        compress="lzw",
        nodata=nodata,
    ) as target:
        target.write(array, 1)
        target.set_band_description(1, description)


def build_daily_kde(
    features_csv: Path,
    boundary_path: Path,
    output_raster: Path,
    config: PipelineConfig,
) -> dict[str, Any]:
    require_modules("numpy", "pandas", "geopandas", "rasterio", "shapely", "pyproj")
    features_csv = Path(features_csv)
    if not features_csv.is_file():
        raise FileNotFoundError(f"找不到特征 CSV：{features_csv}")
    boundary_wgs84, _ = load_union_geometry(
        boundary_path, config.geographic_crs
    )
    boundary_projected, _ = load_union_geometry(
        boundary_path, config.projected_crs
    )
    grid = make_kde_grid(
        boundary_projected,
        cell_size=config.kde_cell_m,
        crs=config.projected_crs,
    )
    accumulator = HistogramAccumulator(
        grid, search_radius_m=config.kde_radius_m
    )
    transformer = pyproj.Transformer.from_crs(
        config.geographic_crs, config.projected_crs, always_xy=True
    )
    read_rows = 0
    inside_rows = 0
    for chunk in pd.read_csv(
        features_csv,
        usecols=["longitude", "latitude"],
        chunksize=config.csv_chunk_rows,
    ):
        read_rows += len(chunk)
        longitudes = pd.to_numeric(chunk["longitude"], errors="coerce").to_numpy()
        latitudes = pd.to_numeric(chunk["latitude"], errors="coerce").to_numpy()
        inside = strict_contains_xy(boundary_wgs84, longitudes, latitudes)
        if inside.any():
            x_values, y_values = transformer.transform(
                longitudes[inside], latitudes[inside]
            )
            accumulator.add(x_values, y_values)
            inside_rows += int(inside.sum())
    density = accumulator.density()
    write_single_band_raster(
        density,
        output_raster,
        grid,
        description="quartic_kde_points_per_square_kilometre",
    )
    return {
        "rows_read": read_rows,
        "points_inside_boundary": inside_rows,
        "points_accumulated": accumulator.n_points_added,
        "shape": density.shape,
    }




**持续热点：** 每天取正密度像元前 3%，叠加得到 0–7 天持续性。


In [ ]:
def _raster_signature(dataset: Any) -> tuple[Any, ...]:
    bounds = tuple(round(float(value), 6) for value in dataset.bounds)
    transform = tuple(round(float(value), 12) for value in dataset.transform)
    return (
        dataset.height,
        dataset.width,
        dataset.crs.to_string() if dataset.crs else None,
        transform,
        bounds,
    )


def build_persistence_raster(
    daily_kde_by_date: Mapping[str, Path],
    output_raster: Path,
    config: PipelineConfig,
) -> dict[str, Any]:
    require_modules("numpy", "rasterio")
    expected_dates = tuple(config.dates)
    if tuple(sorted(daily_kde_by_date)) != tuple(sorted(expected_dates)):
        raise ValueError(
            f"持续性分析必须恰好提供 7 个固定日期：{expected_dates}；"
            f"实际为 {tuple(sorted(daily_kde_by_date))}"
        )
    masks: list[Any] = []
    normalized_arrays: list[Any] = []
    reference_profile = None
    reference_signature = None
    thresholds: dict[str, float] = {}

    for day_text in expected_dates:
        raster_path = Path(daily_kde_by_date[day_text])
        if not raster_path.is_file():
            raise FileNotFoundError(f"缺少逐日 KDE：{day_text} -> {raster_path}")
        with rasterio.open(raster_path) as source:
            signature = _raster_signature(source)
            if reference_signature is None:
                reference_signature = signature
                reference_profile = source.profile.copy()
            elif signature != reference_signature:
                raise ValueError(
                    f"逐日 KDE 网格不一致：{day_text} 的 shape/CRS/transform/extent 不匹配"
                )
            density = source.read(1).astype("float64")
        valid = np.isfinite(density) & (density > 0)
        if not valid.any():
            raise ValueError(f"{day_text} 的 KDE 没有正密度像元")
        percentile = 100.0 * (1.0 - config.daily_hotspot_top_fraction)
        threshold = float(np.percentile(density[valid], percentile))
        thresholds[day_text] = threshold
        daily_mask = valid & (density >= threshold)
        masks.append(daily_mask)
        normalized = np.zeros_like(density, dtype="float64")
        normalized[valid] = density[valid] / threshold if threshold > 0 else 0.0
        normalized_arrays.append(normalized)

    persistence = np.sum(np.stack(masks, axis=0), axis=0).astype("int16")
    mean_normalized = np.mean(
        np.stack(normalized_arrays, axis=0), axis=0
    ).astype("float32")
    output_raster = Path(output_raster)
    if output_raster.exists():
        raise FileExistsError(f"输出已存在：{output_raster}")
    output_raster.parent.mkdir(parents=True, exist_ok=True)
    profile = reference_profile
    profile.update(
        driver="GTiff",
        count=2,
        dtype="float32",
        compress="lzw",
        nodata=0.0,
    )
    with rasterio.open(output_raster, "w", **profile) as target:
        target.write(persistence.astype("float32"), 1)
        target.write(mean_normalized, 2)
        target.set_band_description(1, "daily_top3pct_persistence_count")
        target.set_band_description(2, "mean_density_normalized_by_daily_p97")
    return {
        "thresholds": thresholds,
        "persistent_cell_count": int(
            (persistence == config.persistence_days).sum()
        ),
    }


NEIGHBOURS_8 = (
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1),
)




In [ ]:
def label_components_8(mask: Any) -> tuple[Any, int]:
    require_modules("numpy")
    mask = np.asarray(mask, dtype=bool)
    labels = np.zeros(mask.shape, dtype="int32")
    component_id = 0
    rows, columns = mask.shape
    for row, column in zip(*np.nonzero(mask)):
        if labels[row, column] != 0:
            continue
        component_id += 1
        labels[row, column] = component_id
        queue = [(int(row), int(column))]
        cursor = 0
        while cursor < len(queue):
            current_row, current_column = queue[cursor]
            cursor += 1
            for row_offset, column_offset in NEIGHBOURS_8:
                next_row = current_row + row_offset
                next_column = current_column + column_offset
                if (
                    0 <= next_row < rows
                    and 0 <= next_column < columns
                    and mask[next_row, next_column]
                    and labels[next_row, next_column] == 0
                ):
                    labels[next_row, next_column] = component_id
                    queue.append((next_row, next_column))
    return labels, component_id


CANDIDATE_COLUMNS = [
    "rank", "district", "area_km2", "n_cells", "mean_persistence",
    "mean_density_norm", "center_lon", "center_lat", "geometry",
]




**候选区：** 聚类持续 7 天的热点，删除少于 3 个像元的连通区域。


In [ ]:
def build_study_area_candidates(
    persistence_raster: Path,
    output_gpkg: Path,
    output_csv: Path,
    config: PipelineConfig,
    *,
    district_path: Path | None = None,
) -> Any:
    require_modules("numpy", "pandas", "geopandas", "rasterio", "shapely", "pyproj")
    persistence_raster = Path(persistence_raster)
    with rasterio.open(persistence_raster) as source:
        persistence = source.read(1)
        mean_normalized = source.read(2)
        transform = source.transform
        source_crs = source.crs
    if source_crs is None:
        raise ValueError("持续性栅格缺少 CRS")

    core_mask = persistence == config.persistence_days
    labels, n_components = label_components_8(core_mask)
    records: list[dict[str, Any]] = []
    transformer = pyproj.Transformer.from_crs(
        source_crs, config.geographic_crs, always_xy=True
    )
    for component_id in range(1, n_components + 1):
        component = labels == component_id
        n_cells = int(component.sum())
        if n_cells < config.min_component_cells:
            continue
        shapes = list(
            rasterio_features.shapes(
                component.astype("uint8"),
                mask=component,
                transform=transform,
            )
        )
        polygons = [
            shapely_geometry.shape(geometry_mapping)
            for geometry_mapping, value in shapes
            if int(value) == 1
        ]
        geometry = shapely_ops.unary_union(polygons)
        center = geometry.centroid
        center_lon, center_lat = transformer.transform(center.x, center.y)
        records.append({
            "district": "",
            "area_km2": float(geometry.area / 1_000_000.0),
            "n_cells": n_cells,
            "mean_persistence": float(persistence[component].mean()),
            "mean_density_norm": float(mean_normalized[component].mean()),
            "center_lon": float(center_lon),
            "center_lat": float(center_lat),
            "geometry": geometry,
        })

    if not records:
        candidates = gpd.GeoDataFrame(
            {column: [] for column in CANDIDATE_COLUMNS if column != "geometry"},
            geometry=[],
            crs=source_crs,
        )
        candidates["rank"] = pd.Series(dtype="int64")
    else:
        candidates = gpd.GeoDataFrame(
            records, geometry="geometry", crs=source_crs
        )
        candidates = candidates.sort_values(
            ["mean_density_norm", "area_km2"],
            ascending=[False, False],
            kind="mergesort",
        ).reset_index(drop=True)
        candidates.insert(0, "rank", np.arange(1, len(candidates) + 1))
        if district_path is not None:
            districts = gpd.read_file(district_path)
            require_crs(districts, "区县边界")
            districts = districts.to_crs(source_crs)
            name_candidates = [
                column for column in districts.columns
                if column != "geometry" and districts[column].dtype == object
            ]
            if name_candidates:
                joined = gpd.sjoin(
                    candidates[["geometry"]],
                    districts[[name_candidates[0], "geometry"]],
                    how="left",
                    predicate="intersects",
                )
                district_map = (
                    joined.groupby(joined.index)[name_candidates[0]]
                    .first()
                    .fillna("")
                )
                candidates["district"] = candidates.index.map(district_map).fillna("")
        candidates = candidates[CANDIDATE_COLUMNS]

    output_gpkg = Path(output_gpkg)
    output_csv = Path(output_csv)
    for path in (output_gpkg, output_csv):
        if path.exists():
            raise FileExistsError(f"输出已存在：{path}")
        path.parent.mkdir(parents=True, exist_ok=True)
    require_modules("pyogrio")
    pyogrio.write_dataframe(
        candidates,
        output_gpkg,
        layer="study_area_candidates",
        driver="GPKG",
    )
    csv_frame = pd.DataFrame(candidates.drop(columns="geometry"))
    csv_frame["geometry"] = candidates.geometry.to_wkt()
    csv_frame.to_csv(output_csv, index=False, encoding="utf-8-sig")
    return candidates


## 3. 拥堵识别

删除长时间零速停车段，并为连续低速轨迹增加拥堵标记。


In [ ]:
from __future__ import annotations

def _segment_ids(frame: Any, config: PipelineConfig) -> Any:
    require_modules("pandas", "numpy")
    timestamp = pd.to_numeric(frame["timestamp"], errors="coerce")
    taxi_change = frame["taxi_id"].astype(str).ne(
        frame["taxi_id"].astype(str).shift()
    )
    bad_time = timestamp.isna()
    delta = timestamp.diff()
    explicit_break = pd.to_numeric(
        frame.get("trip_break", 0), errors="coerce"
    ).fillna(0).astype(bool)
    boundary = (
        taxi_change
        | bad_time
        | explicit_break
        | delta.le(0)
        | delta.gt(config.max_gap_sec)
    )
    return boundary.cumsum()




**停车清理：** 删除持续至少 60 分钟的零速段，并重新标记轨迹断点。


In [ ]:
def drop_long_zero_speed_runs(
    frame: Any,
    config: PipelineConfig,
    *,
    speed_column: str | None = None,
) -> tuple[Any, dict[str, int]]:
    require_modules("pandas", "numpy")
    speed_column = speed_column or config.speed_column
    require_columns(
        frame,
        ["taxi_id", "timestamp", "trip_break", speed_column],
        "停车删除输入",
    )
    result = frame.copy()
    result["timestamp"] = pd.to_numeric(result["timestamp"], errors="coerce")
    result = result.sort_values(
        ["taxi_id", "timestamp"], kind="mergesort", na_position="last"
    ).reset_index(drop=True)
    speed = pd.to_numeric(result[speed_column], errors="coerce")
    zero = speed.eq(config.parking_zero_speed_kmh)
    base_segment = _segment_ids(result, config)
    zero_run_boundary = (
        ~zero
        | zero.ne(zero.shift(fill_value=False))
        | base_segment.ne(base_segment.shift(fill_value=-1))
    )
    zero_run_id = zero_run_boundary.cumsum()

    remove = pd.Series(False, index=result.index)
    removed_runs = 0
    for _, index_values in result.loc[zero].groupby(
        zero_run_id.loc[zero], sort=False
    ).groups.items():
        indices = list(index_values)
        first_index = indices[0]
        last_index = indices[-1]
        start_time = result.at[first_index, "timestamp"]
        # 段后第一条同车、同原轨迹段的非零记录可作为零速持续到何时的观察上界。
        end_time = result.at[last_index, "timestamp"]
        next_index = last_index + 1
        if (
            next_index < len(result)
            and result.at[next_index, "taxi_id"] == result.at[last_index, "taxi_id"]
            and base_segment.iat[next_index] == base_segment.iat[last_index]
        ):
            end_time = result.at[next_index, "timestamp"]
        duration = (
            float(end_time - start_time)
            if pd.notna(start_time) and pd.notna(end_time)
            else -1.0
        )
        if duration >= config.parking_min_duration_sec:
            remove.loc[indices] = True
            removed_runs += 1

    removed_indices = set(result.index[remove].tolist())
    retained = result.loc[~remove].copy()
    if len(retained):
        original_indices = retained.index.to_numpy()
        break_after_deleted = np.zeros(len(retained), dtype=bool)
        for position, original_index in enumerate(original_indices):
            if position == 0:
                break_after_deleted[position] = True
                continue
            previous_original = original_indices[position - 1]
            if original_index - previous_original > 1:
                between = range(previous_original + 1, original_index)
                if any(index in removed_indices for index in between):
                    break_after_deleted[position] = True
        retained["trip_break"] = np.maximum(
            pd.to_numeric(retained["trip_break"], errors="coerce")
            .fillna(1)
            .astype("int8")
            .to_numpy(),
            break_after_deleted.astype("int8"),
        )
    retained = retained.reset_index(drop=True)
    return retained, {
        "rows_input": len(result),
        "rows_removed": int(remove.sum()),
        "long_zero_speed_runs_removed": removed_runs,
        "rows_output": len(retained),
    }




**拥堵判定：** 连续低于 20 km/h 且持续至少 4 分钟时标记拥堵。


In [ ]:
def label_congestion(
    frame: Any,
    config: PipelineConfig,
    *,
    speed_column: str | None = None,
) -> tuple[Any, dict[str, int]]:
    require_modules("pandas", "numpy")
    speed_column = speed_column or config.speed_column
    require_columns(
        frame,
        ["taxi_id", "timestamp", "trip_break", "bad_coord", speed_column],
        "拥堵标注输入",
    )
    result = frame.copy().reset_index(drop=True)
    timestamp = pd.to_numeric(result["timestamp"], errors="coerce")
    speed = pd.to_numeric(result[speed_column], errors="coerce")
    low_speed = speed.lt(config.congestion_speed_kmh) & speed.notna()
    bad_coord = pd.to_numeric(
        result["bad_coord"], errors="coerce"
    ).fillna(1).astype(bool)
    base_segment = _segment_ids(result, config)
    low_speed &= ~bad_coord & timestamp.notna()
    run_boundary = (
        ~low_speed
        | low_speed.ne(low_speed.shift(fill_value=False))
        | base_segment.ne(base_segment.shift(fill_value=-1))
    )
    run_id = run_boundary.cumsum()
    congestion = np.zeros(len(result), dtype="int8")
    congested_runs = 0
    for _, index_values in result.loc[low_speed].groupby(
        run_id.loc[low_speed], sort=False
    ).groups.items():
        indices = list(index_values)
        start_time = timestamp.loc[indices[0]]
        end_time = timestamp.loc[indices[-1]]
        duration = float(end_time - start_time)
        if duration >= config.congestion_min_duration_sec:
            congestion[indices] = 1
            congested_runs += 1
    result["congestion"] = congestion
    return result, {
        "rows": len(result),
        "congested_rows": int(congestion.sum()),
        "congested_runs": congested_runs,
    }


def clean_and_label_vehicle_frame(
    vehicle_frame: Any,
    config: PipelineConfig,
) -> tuple[Any, dict[str, int]]:
    cleaned, parking_metrics = drop_long_zero_speed_runs(vehicle_frame, config)
    labelled, congestion_metrics = label_congestion(cleaned, config)
    # 删除后重新生成事件，且不跨 trip_break。
    occupied = pd.to_numeric(labelled["occupied"], errors="coerce").fillna(0).astype("int8")
    previous = occupied.shift()
    continuous = labelled["trip_break"].eq(0)
    labelled["event_type"] = ""
    labelled.loc[continuous & previous.eq(0) & occupied.eq(1), "event_type"] = "pickup"
    labelled.loc[continuous & previous.eq(1) & occupied.eq(0), "event_type"] = "dropoff"
    return labelled, {**parking_metrics, **congestion_metrics}




In [ ]:
def clean_and_label_csv(
    input_csv: Path,
    output_csv: Path,
    config: PipelineConfig,
) -> dict[str, int]:
    require_modules("pandas", "numpy")
    input_csv = Path(input_csv)
    output_csv = Path(output_csv)
    if not input_csv.is_file():
        raise FileNotFoundError(f"找不到特征 CSV：{input_csv}")
    if output_csv.exists():
        raise FileExistsError(f"输出已存在：{output_csv}")
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    metrics = {
        "vehicles": 0,
        "rows_input": 0,
        "rows_removed": 0,
        "rows_output": 0,
        "long_zero_speed_runs_removed": 0,
        "congested_rows": 0,
        "congested_runs": 0,
    }
    first_write = True
    for vehicle in _iter_complete_vehicle_blocks(
        input_csv, chunksize=config.csv_chunk_rows
    ):
        labelled, vehicle_metrics = clean_and_label_vehicle_frame(vehicle, config)
        labelled.to_csv(
            output_csv,
            mode="x" if first_write else "a",
            header=first_write,
            index=False,
            encoding="utf-8",
        )
        first_write = False
        metrics["vehicles"] += 1
        for key in metrics:
            if key != "vehicles" and key in vehicle_metrics:
                metrics[key] += int(vehicle_metrics[key])
    if first_write:
        pd.DataFrame(columns=FEATURE_COLUMNS + ["congestion"]).to_csv(
            output_csv, mode="x", index=False, encoding="utf-8"
        )
    return metrics


## 4. OD 轨迹提取

筛选研究区相关车辆，并生成 OD、完整轨迹点和轨迹线。


In [ ]:
from __future__ import annotations

TRIP_COLUMNS = [
    "route_id", "trip_date", "trip_id", "taxi_id",
    "start_time", "start_lon", "start_lat",
    "end_time", "end_lon", "end_lat",
    "duration_sec", "straight_line_km", "n_points",
]
INCOMPLETE_COLUMNS = [
    "route_id", "trip_date", "trip_id", "taxi_id",
    "start_time", "start_lon", "start_lat", "reason",
]
TRIP_POINT_COLUMNS = [
    "route_id", "trip_date", "trip_id", "taxi_id", "seq", "is_endpoint",
    "timestamp", "longitude", "latitude", "occupied", "congestion", "geometry",
]


def canonical_day_id(day_text: str) -> str:
    parsed = datetime.strptime(day_text, "%Y-%m-%d")
    return parsed.strftime("%Y%m%d")


def find_qualified_taxi_ids(
    frame: Any,
    study_area_wgs84: Any,
) -> set[str]:
    require_modules("pandas", "numpy", "shapely")
    require_columns(
        frame,
        ["taxi_id", "longitude", "latitude", "occupied"],
        "研究区车辆筛选输入",
    )
    longitude = pd.to_numeric(frame["longitude"], errors="coerce").to_numpy()
    latitude = pd.to_numeric(frame["latitude"], errors="coerce").to_numpy()
    occupied = pd.to_numeric(
        frame["occupied"], errors="coerce"
    ).fillna(0).to_numpy()
    inside = strict_contains_xy(study_area_wgs84, longitude, latitude)
    hit = inside & (occupied == 1)
    return set(frame.loc[hit, "taxi_id"].astype(str).unique())


def find_qualified_taxi_ids_csv(
    input_csv: Path,
    study_area_wgs84: Any,
    config: PipelineConfig,
) -> tuple[set[str], int]:
    require_modules("pandas")
    qualified: set[str] = set()
    total_rows = 0
    for chunk in pd.read_csv(
        input_csv,
        usecols=["taxi_id", "longitude", "latitude", "occupied"],
        dtype={"taxi_id": str},
        chunksize=config.csv_chunk_rows,
    ):
        total_rows += len(chunk)
        qualified.update(find_qualified_taxi_ids(chunk, study_area_wgs84))
    return qualified, total_rows




In [ ]:
def write_qualified_taxi_day_csv(
    input_csv: Path,
    output_csv: Path,
    qualified_ids: set[str],
    config: PipelineConfig,
) -> int:
    require_modules("pandas", "numpy")
    input_csv = Path(input_csv)
    output_csv = Path(output_csv)
    if output_csv.exists():
        raise FileExistsError(f"输出已存在：{output_csv}")
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    written = 0
    first_write = True
    qualified_ids = {str(value) for value in qualified_ids}
    for vehicle in _iter_complete_vehicle_blocks(
        input_csv, chunksize=config.csv_chunk_rows
    ):
        if vehicle.empty or str(vehicle.iloc[0]["taxi_id"]) not in qualified_ids:
            continue
        keep = vehicle.copy()
        keep["taxi_id"] = keep["taxi_id"].astype(str)
        if "bad_coord" in keep:
            source_sequence = np.arange(len(keep), dtype="int64")
            valid = pd.to_numeric(
                keep["bad_coord"], errors="coerce"
            ).fillna(1).eq(0).to_numpy()
            keep = keep.loc[valid].copy()
            retained_sequence = source_sequence[valid]
            gap_after_removed = np.r_[
                True, np.diff(retained_sequence) != 1
            ] if len(retained_sequence) else np.array([], dtype=bool)
            if len(keep):
                existing_break = pd.to_numeric(
                    keep["trip_break"], errors="coerce"
                ).fillna(1).astype("int8")
                keep["trip_break"] = np.maximum(
                    existing_break.to_numpy(),
                    gap_after_removed.astype("int8"),
                )
        if keep.empty:
            continue
        keep.to_csv(
            output_csv,
            mode="x" if first_write else "a",
            header=first_write,
            index=False,
            encoding="utf-8",
        )
        first_write = False
        written += len(keep)
    if first_write:
        pd.read_csv(input_csv, nrows=0).to_csv(
            output_csv, mode="x", index=False, encoding="utf-8"
        )
    return written


def filter_vehicle_day_by_study_area(
    input_csv: Path,
    study_area_path: Path,
    output_csv: Path,
    qualified_ids_csv: Path,
    config: PipelineConfig,
) -> dict[str, int]:
    require_modules("pandas", "geopandas", "shapely")
    study_area, _ = load_union_geometry(
        study_area_path, config.geographic_crs
    )
    qualified_ids, total_rows = find_qualified_taxi_ids_csv(
        input_csv, study_area, config
    )
    qualified_ids_csv = Path(qualified_ids_csv)
    if qualified_ids_csv.exists():
        raise FileExistsError(f"输出已存在：{qualified_ids_csv}")
    qualified_ids_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"taxi_id": sorted(qualified_ids)}).to_csv(
        qualified_ids_csv, index=False, encoding="utf-8-sig"
    )
    rows_written = write_qualified_taxi_day_csv(
        input_csv, output_csv, qualified_ids, config
    )
    return {
        "rows_scanned": total_rows,
        "qualified_taxis": len(qualified_ids),
        "rows_written": rows_written,
    }




In [ ]:
def find_strict_od_runs(vehicle_frame: Any) -> list[tuple[int, int | None]]:
    require_modules("pandas", "numpy")
    require_columns(
        vehicle_frame,
        ["occupied", "congestion", "trip_break"],
        "OD 状态机输入",
    )
    occupied = (
        pd.to_numeric(vehicle_frame["occupied"], errors="coerce")
        .fillna(0)
        .eq(1)
        .to_numpy()
    )
    congestion = (
        pd.to_numeric(vehicle_frame["congestion"], errors="coerce")
        .fillna(0)
        .eq(1)
        .to_numpy()
    )
    trip_break = (
        pd.to_numeric(vehicle_frame["trip_break"], errors="coerce")
        .fillna(1)
        .eq(1)
        .to_numpy()
    )
    results: list[tuple[int, int | None]] = []
    n_rows = len(vehicle_frame)
    block_start = 0
    while block_start < n_rows:
        block_end = block_start + 1
        while block_end < n_rows and not trip_break[block_end]:
            block_end += 1
        cursor = block_start
        while cursor < block_end:
            if not occupied[cursor]:
                cursor += 1
                continue
            run_start = cursor
            while cursor < block_end and occupied[cursor]:
                cursor += 1
            run_end_exclusive = cursor
            if (
                run_start > block_start
                and not occupied[run_start - 1]
                and congestion[run_start - 1]
            ):
                destination = (
                    run_end_exclusive if run_end_exclusive < block_end else None
                )
                results.append((run_start, destination))
        block_start = block_end
    return results


def _empty_frame(columns: Sequence[str]) -> Any:
    require_modules("pandas")
    return pd.DataFrame({column: pd.Series(dtype="object") for column in columns})




**OD 规则：** O 为拥堵后首个载客点，D 为载客段后的首个非载客点。


In [ ]:
def extract_strict_congestion_od(
    qualified_frame: Any,
    day_text: str,
    *,
    first_trip_id: int = 1,
) -> tuple[Any, Any, Any, int]:
    require_modules("pandas", "numpy")
    required = [
        "taxi_id", "timestamp", "longitude", "latitude",
        "occupied", "congestion", "trip_break", "bad_coord",
    ]
    require_columns(qualified_frame, required, "合格车辆 OD 输入")
    day_id = canonical_day_id(day_text)
    frame = qualified_frame.copy()
    frame["taxi_id"] = frame["taxi_id"].astype(str)
    frame["timestamp"] = pd.to_numeric(frame["timestamp"], errors="coerce")
    frame = frame.sort_values(
        ["taxi_id", "timestamp"], kind="mergesort", na_position="last"
    ).reset_index(drop=True)
    # 先在包含坏坐标行的完整车序列中编号，再删除坏行。编号出现缺口时，
    # 将其后的第一条有效记录强制设为断点，禁止状态机跨被删位置拼接。
    frame["_source_sequence"] = frame.groupby(
        "taxi_id", sort=False
    ).cumcount()
    valid_row = (
        frame["timestamp"].notna()
        & pd.to_numeric(frame["bad_coord"], errors="coerce").fillna(1).eq(0)
    )
    frame = frame.loc[valid_row].copy()
    source_gap = frame.groupby("taxi_id", sort=False)[
        "_source_sequence"
    ].diff().ne(1)
    existing_break = pd.to_numeric(
        frame["trip_break"], errors="coerce"
    ).fillna(1).astype("int8")
    frame["trip_break"] = np.maximum(
        existing_break.to_numpy(),
        source_gap.fillna(True).astype("int8").to_numpy(),
    )
    frame = frame.drop(columns="_source_sequence").reset_index(drop=True)

    trip_records: list[dict[str, Any]] = []
    incomplete_records: list[dict[str, Any]] = []
    point_records: list[dict[str, Any]] = []
    next_trip_id = int(first_trip_id)

    for taxi_id, vehicle in frame.groupby("taxi_id", sort=False):
        vehicle = vehicle.reset_index(drop=True)
        for start_index, end_index in find_strict_od_runs(vehicle):
            trip_id = next_trip_id
            next_trip_id += 1
            route_id = f"{day_id}_{trip_id}"
            origin = vehicle.iloc[start_index]
            if end_index is None:
                incomplete_records.append({
                    "route_id": route_id,
                    "trip_date": day_text,
                    "trip_id": trip_id,
                    "taxi_id": taxi_id,
                    "start_time": origin["timestamp"],
                    "start_lon": origin["longitude"],
                    "start_lat": origin["latitude"],
                    "reason": "no_observed_dropoff_before_break_or_vehicle_end",
                })
                continue

            destination = vehicle.iloc[end_index]
            duration_sec = float(
                destination["timestamp"] - origin["timestamp"]
            )
            straight_line_km = haversine_m(
                float(origin["latitude"]),
                float(origin["longitude"]),
                float(destination["latitude"]),
                float(destination["longitude"]),
            ) / 1_000.0
            trajectory = vehicle.iloc[start_index: end_index + 1]
            trip_records.append({
                "route_id": route_id,
                "trip_date": day_text,
                "trip_id": trip_id,
                "taxi_id": taxi_id,
                "start_time": origin["timestamp"],
                "start_lon": origin["longitude"],
                "start_lat": origin["latitude"],
                "end_time": destination["timestamp"],
                "end_lon": destination["longitude"],
                "end_lat": destination["latitude"],
                "duration_sec": duration_sec,
                "straight_line_km": straight_line_km,
                "n_points": len(trajectory),
            })
            for sequence, (_, point) in enumerate(trajectory.iterrows()):
                point_records.append({
                    "route_id": route_id,
                    "trip_date": day_text,
                    "trip_id": trip_id,
                    "taxi_id": taxi_id,
                    "seq": sequence,
                    "is_endpoint": int(
                        sequence == 0 or sequence == len(trajectory) - 1
                    ),
                    "timestamp": point["timestamp"],
                    "longitude": point["longitude"],
                    "latitude": point["latitude"],
                    "occupied": point["occupied"],
                    "congestion": point["congestion"],
                })

    trips = (
        pd.DataFrame(trip_records, columns=TRIP_COLUMNS)
        if trip_records else _empty_frame(TRIP_COLUMNS)
    )
    incomplete = (
        pd.DataFrame(incomplete_records, columns=INCOMPLETE_COLUMNS)
        if incomplete_records else _empty_frame(INCOMPLETE_COLUMNS)
    )
    point_columns_no_geometry = [
        column for column in TRIP_POINT_COLUMNS if column != "geometry"
    ]
    points = (
        pd.DataFrame(point_records, columns=point_columns_no_geometry)
        if point_records else _empty_frame(point_columns_no_geometry)
    )
    return trips, incomplete, points, next_trip_id




In [ ]:
def trip_points_to_geodataframe(points: Any, config: PipelineConfig) -> Any:
    require_modules("pandas", "geopandas", "shapely")
    require_columns(
        points,
        [column for column in TRIP_POINT_COLUMNS if column != "geometry"],
        "行程点",
    )
    geometry = gpd.points_from_xy(
        pd.to_numeric(points["longitude"], errors="coerce"),
        pd.to_numeric(points["latitude"], errors="coerce"),
        crs=config.geographic_crs,
    )
    return gpd.GeoDataFrame(points.copy(), geometry=geometry, crs=config.geographic_crs)


def trip_points_to_lines(
    trips: Any,
    points: Any,
    config: PipelineConfig,
) -> Any:
    require_modules("pandas", "geopandas", "shapely")
    require_columns(trips, TRIP_COLUMNS, "行程摘要")
    require_columns(
        points,
        ["route_id", "seq", "longitude", "latitude"],
        "行程点",
    )
    geometries: dict[str, Any] = {}
    for route_id, group in points.groupby("route_id", sort=False):
        ordered = group.sort_values("seq", kind="mergesort")
        coordinates = list(zip(
            pd.to_numeric(ordered["longitude"], errors="coerce"),
            pd.to_numeric(ordered["latitude"], errors="coerce"),
        ))
        valid_coordinates = [
            (float(x), float(y))
            for x, y in coordinates
            if pd.notna(x) and pd.notna(y)
        ]
        geometries[str(route_id)] = (
            shapely_geometry.LineString(valid_coordinates)
            if len(valid_coordinates) >= 2 else None
        )
    line_frame = trips.copy()
    line_frame["geometry"] = line_frame["route_id"].astype(str).map(geometries)
    return gpd.GeoDataFrame(
        line_frame, geometry="geometry", crs=config.geographic_crs
    )


def write_gpkg_layer(
    frame: Any,
    output_path: Path,
    *,
    layer: str,
) -> None:
    require_modules("geopandas", "pyogrio")
    require_crs(frame, f"GPKG 图层 {layer}")
    output_path = Path(output_path)
    if output_path.exists():
        raise FileExistsError(f"输出已存在：{output_path}")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    pyogrio.write_dataframe(
        frame,
        output_path,
        layer=layer,
        driver="GPKG",
    )


def export_od_outputs(
    trips: Any,
    incomplete: Any,
    points: Any,
    output_dir: Path,
    config: PipelineConfig,
) -> dict[str, Path]:
    require_modules("pandas", "geopandas", "shapely")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = {
        "trips": output_dir / "trips.csv",
        "incomplete": output_dir / "incomplete_trips.csv",
        "points": output_dir / "trip_points.gpkg",
        "lines": output_dir / "trip_lines.gpkg",
    }
    for path in paths.values():
        if path.exists():
            raise FileExistsError(f"输出已存在：{path}")
    trips.reindex(columns=TRIP_COLUMNS).to_csv(
        paths["trips"], index=False, encoding="utf-8-sig"
    )
    incomplete.reindex(columns=INCOMPLETE_COLUMNS).to_csv(
        paths["incomplete"], index=False, encoding="utf-8-sig"
    )
    point_gdf = trip_points_to_geodataframe(points, config)
    line_gdf = trip_points_to_lines(trips, points, config)
    write_gpkg_layer(point_gdf, paths["points"], layer="trip_points")
    write_gpkg_layer(line_gdf, paths["lines"], layer="trip_lines")
    return paths


## 5. 道路等级与时段匹配

为轨迹点匹配最近道路等级，并按北京时间标注早高峰、晚高峰和平峰。


In [ ]:
from __future__ import annotations

def unix_seconds_to_beijing(
    values: Any,
    *,
    timezone_name: str = "Asia/Shanghai",
) -> Any:
    require_modules("pandas")
    numeric = pd.to_numeric(values, errors="coerce")
    return pd.to_datetime(
        numeric, unit="s", utc=True, errors="coerce"
    ).dt.tz_convert(timezone_name)


def classify_time_period(
    timestamps: Any,
    config: PipelineConfig,
) -> Any:
    require_modules("pandas", "numpy")
    local_time = unix_seconds_to_beijing(
        timestamps, timezone_name=config.timezone_name
    )
    minutes = local_time.dt.hour * 60 + local_time.dt.minute

    def parse_hhmm(value: str) -> int:
        hour, minute = (int(part) for part in value.split(":"))
        return hour * 60 + minute

    morning = minutes.ge(parse_hhmm(config.morning_start)) & minutes.lt(
        parse_hhmm(config.morning_end)
    )
    evening = minutes.ge(parse_hhmm(config.evening_start)) & minutes.lt(
        parse_hhmm(config.evening_end)
    )
    labels = np.select(
        [morning.fillna(False), evening.fillna(False)],
        ["morning_peak", "evening_peak"],
        default="off_peak",
    ).astype(object)
    labels[local_time.isna().to_numpy()] = "invalid_time"
    return pd.Series(labels, index=getattr(timestamps, "index", None), dtype="object")


def validate_classification_roads(
    roads: Any,
    config: PipelineConfig,
) -> Any:
    require_modules("geopandas")
    require_columns(
        roads,
        [config.road_class_field, "geometry"],
        "道路等级路网",
    )
    require_crs(roads, "道路等级路网")
    clean = roads.loc[
        roads.geometry.notna() & ~roads.geometry.is_empty
    ].copy()
    if clean.empty:
        raise ValueError("道路等级路网没有有效几何")
    stable_fields = [
        field for field in ("osm_id", config.road_class_field)
        if field in clean.columns
    ]
    if stable_fields:
        clean = clean.sort_values(stable_fields, kind="mergesort")
    clean = clean.explode(index_parts=False, ignore_index=True)
    return clean.to_crs(config.projected_crs).reset_index(drop=True)


def _nearest_tree_index(
    tree: Any,
    point: Any,
    *,
    max_distance: float,
) -> tuple[int | None, float | None]:
    if not hasattr(tree, "query_nearest"):
        raise RuntimeError("道路等级匹配要求 Shapely 2.x 的 STRtree.query_nearest")
    indices, distances = tree.query_nearest(
        point,
        max_distance=float(max_distance),
        return_distance=True,
        all_matches=True,
    )
    indices = np.asarray(indices).reshape(-1)
    distances = np.asarray(distances, dtype="float64").reshape(-1)
    if len(indices) == 0:
        return None, None
    order = np.lexsort((indices.astype("int64"), distances))
    selected = int(order[0])
    return int(indices[selected]), float(distances[selected])




**道路匹配：** 在 30 米内使用 STRtree 查找唯一最近道路。


In [ ]:
def match_road_types(
    points: Any,
    roads: Any,
    config: PipelineConfig,
    *,
    lon_col: str = "longitude",
    lat_col: str = "latitude",
    timestamp_col: str = "timestamp",
) -> Any:
    require_modules("numpy", "pandas", "geopandas", "shapely")
    require_columns(points, [lon_col, lat_col, timestamp_col], "道路等级匹配点")
    projected_roads = validate_classification_roads(roads, config)
    road_geometries = list(projected_roads.geometry)
    tree = shapely_strtree.STRtree(road_geometries)

    work = points.copy().reset_index(drop=True)
    longitude = pd.to_numeric(work[lon_col], errors="coerce")
    latitude = pd.to_numeric(work[lat_col], errors="coerce")
    valid_coordinate = (
        longitude.between(-180, 180)
        & latitude.between(-90, 90)
        & longitude.notna()
        & latitude.notna()
    )
    point_gdf = gpd.GeoDataFrame(
        work,
        geometry=gpd.points_from_xy(longitude, latitude),
        crs=config.geographic_crs,
    )
    projected_points = point_gdf.to_crs(config.projected_crs)

    road_type: list[Any] = []
    road_distance: list[float] = []
    match_status: list[str] = []
    for index, point in enumerate(projected_points.geometry):
        if not bool(valid_coordinate.iat[index]) or point is None or point.is_empty:
            road_type.append(None)
            road_distance.append(float("nan"))
            match_status.append("invalid_coord")
            continue
        road_index, distance = _nearest_tree_index(
            tree,
            point,
            max_distance=config.road_match_tolerance_m,
        )
        if road_index is None:
            road_type.append(None)
            road_distance.append(float("nan"))
            match_status.append("unmatched")
        else:
            road_type.append(
                projected_roads.iloc[road_index][config.road_class_field]
            )
            road_distance.append(float(distance))
            match_status.append("matched")

    point_gdf["road_type"] = road_type
    point_gdf["road_distance_m"] = road_distance
    point_gdf["road_match_status"] = match_status
    point_gdf["time_period"] = classify_time_period(
        point_gdf[timestamp_col], config
    ).to_numpy()
    return point_gdf


def load_and_match_road_types(
    points: Any,
    road_path: Path,
    config: PipelineConfig,
) -> Any:
    require_modules("geopandas")
    road_path = Path(road_path)
    if not road_path.exists():
        raise FileNotFoundError(f"找不到道路等级路网：{road_path}")
    roads = gpd.read_file(road_path)
    return match_road_types(points, roads, config)


## 6. 最优路径计算

使用详细单行路网计算严格最短路和一条符合限制的替代路径。


In [ ]:
from __future__ import annotations

DETAILED_ROAD_FIELDS = {
    "osm_id",
    "fclass",
    "oneway",
    "layer",
    "bridge",
    "tunnel",
    "geometry",
}

DEFAULT_DRIVABLE_CLASSES = frozenset(
    {
        "motorway",
        "motorway_link",
        "trunk",
        "trunk_link",
        "primary",
        "primary_link",
        "secondary",
        "secondary_link",
        "tertiary",
        "tertiary_link",
        "residential",
        "unclassified",
        "living_street",
        "service",
    }
)


def _normalise_oneway(value: Any) -> str:
    """Return B (both), F (stored direction), or T (reverse direction)."""
    if pd.isna(value):
        return "B"
    code = str(value).strip().upper()
    if code in {"", "NONE", "NAN", "B"}:
        return "B"
    if code in {"F", "T"}:
        return code
    raise ValueError(f"未知 oneway 编码: {value!r}；仅支持 B/F/T/空值")


def _reverse_line(line: shapely_geometry.LineString) -> shapely_geometry.LineString:
    return shapely_geometry.LineString(list(line.coords)[::-1])


def _node_key(coord: Iterable[float], precision: int = 3) -> tuple[float, float]:
    x, y = coord
    return round(float(x), precision), round(float(y), precision)




In [ ]:
def validate_detailed_roads(
    roads: gpd.GeoDataFrame,
    *,
    projected_crs: str = "EPSG:32650",
    drivable_classes: frozenset[str] = DEFAULT_DRIVABLE_CLASSES,
) -> gpd.GeoDataFrame:
    """Validate, filter, project, and split a detailed one-line road network."""
    missing = DETAILED_ROAD_FIELDS.difference(roads.columns)
    if missing:
        raise KeyError(f"详细路网缺少字段: {sorted(missing)}")
    if roads.crs is None:
        raise ValueError("详细路网缺少 CRS，不能安全计算米制距离")

    clean = roads.copy()
    clean["fclass"] = clean["fclass"].astype("string").str.strip().str.lower()
    clean = clean.loc[
        clean.geometry.notna()
        & ~clean.geometry.is_empty
        & clean["fclass"].isin(drivable_classes)
    ].copy()
    if clean.empty:
        raise ValueError("详细路网中没有可用于机动车路由的线")

    clean = clean.explode(index_parts=False, ignore_index=True)
    clean = clean.loc[clean.geom_type.eq("LineString")].copy()
    clean = clean.to_crs(projected_crs)
    clean["oneway"] = clean["oneway"].map(_normalise_oneway)
    clean["_stable_osm"] = clean["osm_id"].astype(str)
    clean["_stable_geometry"] = clean.geometry.map(lambda value: value.wkb_hex)
    clean = clean.sort_values(
        ["_stable_osm", "_stable_geometry"], kind="mergesort"
    ).reset_index(drop=True)

    records: list[dict[str, Any]] = []
    for row in clean.itertuples(index=False):
        coords = list(row.geometry.coords)
        for start, end in zip(coords[:-1], coords[1:]):
            segment = shapely_geometry.LineString([start, end])
            if not math.isfinite(segment.length) or segment.length <= 0:
                continue
            records.append(
                {
                    "osm_id": str(row.osm_id),
                    "fclass": str(row.fclass),
                    "oneway": str(row.oneway),
                    "layer": row.layer,
                    "bridge": row.bridge,
                    "tunnel": row.tunnel,
                    "length_m": float(segment.length),
                    "geometry": segment,
                }
            )
    if not records:
        raise ValueError("详细路网拆分后没有正长度线段")

    segments = gpd.GeoDataFrame(records, geometry="geometry", crs=projected_crs)
    segments["road_idx"] = np.arange(len(segments), dtype=np.int64)
    return segments


def load_detailed_roads(
    path: str | Path,
    *,
    projected_crs: str = "EPSG:32650",
    drivable_classes: frozenset[str] = DEFAULT_DRIVABLE_CLASSES,
) -> gpd.GeoDataFrame:
    """Read a GPKG/other GeoPandas source, then enforce the detailed-road contract."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"详细路网不存在: {path}")
    return validate_detailed_roads(
        gpd.read_file(path),
        projected_crs=projected_crs,
        drivable_classes=drivable_classes,
    )


def _build_multidigraph(segments: gpd.GeoDataFrame) -> nx.MultiDiGraph:
    graph = nx.MultiDiGraph()
    for row in segments.itertuples(index=False):
        start, end = list(row.geometry.coords)
        u, v = _node_key(start), _node_key(end)
        common = {
            "length": float(row.length_m),
            "road_idx": int(row.road_idx),
            "osm_id": str(row.osm_id),
            "fclass": str(row.fclass),
            "layer": row.layer,
            "bridge": row.bridge,
            "tunnel": row.tunnel,
        }
        if row.oneway in {"B", "F"}:
            graph.add_edge(u, v, geometry=row.geometry, **common)
        if row.oneway in {"B", "T"}:
            graph.add_edge(v, u, geometry=_reverse_line(row.geometry), **common)
    return graph




**路网构图：** 按单行方向构建有向图，只保留最大弱连通分量。


In [ ]:
def _simplify_graph(graph: nx.MultiDiGraph) -> nx.DiGraph:
    """Keep the shortest legal parallel edge for every directed node pair."""
    simple = nx.DiGraph()
    for u, v, attrs in graph.edges(data=True):
        current = simple.get_edge_data(u, v)
        if current is None or float(attrs["length"]) < float(current["length"]):
            simple.add_edge(u, v, **attrs)
    return simple


def prepare_routing_network(
    road_segments: gpd.GeoDataFrame,
) -> tuple[gpd.GeoDataFrame, nx.DiGraph]:
    """Retain the largest weak component and return aligned roads plus a DiGraph."""
    initial = _build_multidigraph(road_segments)
    if initial.number_of_edges() == 0:
        raise ValueError("详细路网不能形成有向边")
    largest_nodes = max(nx.weakly_connected_components(initial), key=len)

    keep = []
    for row in road_segments.itertuples(index=False):
        start, end = list(row.geometry.coords)
        keep.append(
            _node_key(start) in largest_nodes and _node_key(end) in largest_nodes
        )
    retained = road_segments.loc[keep].copy().reset_index(drop=True)
    if retained.empty:
        raise ValueError("详细路网最大弱连通分量为空")
    retained["road_idx"] = np.arange(len(retained), dtype=np.int64)

    graph = _simplify_graph(_build_multidigraph(retained))
    if graph.number_of_edges() == 0:
        raise ValueError("详细路网最大弱连通分量不能形成路由图")
    return retained, graph


def make_od_points(
    trips: pd.DataFrame,
    *,
    projected_crs: str = "EPSG:32650",
) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame, set[int]]:
    """Create valid WGS84 O/D points; return invalid row positions separately."""
    required = {"route_id", "start_lon", "start_lat", "end_lon", "end_lat"}
    missing = required.difference(trips.columns)
    if missing:
        raise KeyError(f"OD 表缺少字段: {sorted(missing)}")
    if trips["route_id"].astype("string").duplicated().any():
        raise ValueError("route_id 必须全局唯一")

    work = trips.reset_index(drop=True).copy()
    numeric = ["start_lon", "start_lat", "end_lon", "end_lat"]
    for column in numeric:
        work[column] = pd.to_numeric(work[column], errors="coerce")
    finite = np.isfinite(work[numeric].to_numpy(dtype=float)).all(axis=1)
    bounds = (
        work["start_lon"].between(-180, 180)
        & work["end_lon"].between(-180, 180)
        & work["start_lat"].between(-90, 90)
        & work["end_lat"].between(-90, 90)
    )
    valid = finite & bounds.to_numpy()
    invalid_positions = set(np.flatnonzero(~valid).tolist())
    valid_work = work.loc[valid].copy()
    valid_work["od_pos"] = valid_work.index.astype(int)

    common = valid_work[["od_pos", "route_id"]].copy()
    starts = gpd.GeoDataFrame(
        common,
        geometry=gpd.points_from_xy(
            valid_work["start_lon"], valid_work["start_lat"], crs="EPSG:4326"
        ),
        crs="EPSG:4326",
    ).to_crs(projected_crs)
    ends = gpd.GeoDataFrame(
        common,
        geometry=gpd.points_from_xy(
            valid_work["end_lon"], valid_work["end_lat"], crs="EPSG:4326"
        ),
        crs="EPSG:4326",
    ).to_crs(projected_crs)
    return starts, ends, invalid_positions




In [ ]:
def snap_to_roads(
    points: gpd.GeoDataFrame,
    roads: gpd.GeoDataFrame,
    *,
    max_snap_m: float = 500.0,
) -> pd.DataFrame:
    """Snap to the deterministic nearest retained road, within max_snap_m."""
    if points.empty:
        return pd.DataFrame(
            columns=["matched", "road_idx", "snap_m", "position", "point"]
        ).rename_axis("od_pos")
    if points.crs != roads.crs:
        points = points.to_crs(roads.crs)
    joined = gpd.sjoin_nearest(
        points,
        roads[["road_idx", "geometry"]],
        how="left",
        max_distance=float(max_snap_m),
        distance_col="snap_m",
    )
    joined = joined.sort_values(
        ["od_pos", "snap_m", "road_idx"], kind="stable", na_position="last"
    ).drop_duplicates("od_pos", keep="first")
    roads_by_idx = roads.set_index("road_idx")

    records = []
    for item in joined.itertuples(index=False):
        base = {"od_pos": int(item.od_pos)}
        if pd.isna(item.road_idx):
            records.append(
                {
                    **base,
                    "matched": False,
                    "road_idx": pd.NA,
                    "snap_m": np.nan,
                    "position": np.nan,
                    "point": None,
                }
            )
            continue
        road_idx = int(item.road_idx)
        line = roads_by_idx.loc[road_idx].geometry
        position = float(line.project(item.geometry))
        records.append(
            {
                **base,
                "matched": True,
                "road_idx": road_idx,
                "snap_m": float(item.snap_m),
                "position": position,
                "point": line.interpolate(position),
            }
        )
    return pd.DataFrame(records).set_index("od_pos", drop=True)


def _road_info(roads_by_idx: gpd.GeoDataFrame, road_idx: int) -> dict[str, Any]:
    row = roads_by_idx.loc[road_idx]
    line = row.geometry
    return {
        "road_idx": int(road_idx),
        "line": line,
        "length": float(line.length),
        "coords": list(line.coords),
        "direction": _normalise_oneway(row.oneway),
    }


def _line_part(
    line: shapely_geometry.LineString,
    start: float,
    end: float,
    *,
    reverse: bool = False,
) -> shapely_geometry.LineString | None:
    if abs(end - start) <= 1e-9:
        return None
    part = shapely_ops.substring(line, min(start, end), max(start, end))
    if part.is_empty or not isinstance(part, shapely_geometry.LineString):
        return None
    return _reverse_line(part) if reverse else part




In [ ]:
def _source_options(
    info: dict[str, Any], position: float
) -> list[
    tuple[
        tuple[float, float],
        float,
        shapely_geometry.LineString | None,
        tuple[tuple[float, float], tuple[float, float]],
    ]
]:
    """Legal moves from a snapped O to either endpoint of its road segment."""
    line = info["line"]
    start_node = _node_key(line.coords[0])
    end_node = _node_key(line.coords[-1])
    options = []
    if info["direction"] in {"B", "T"}:
        options.append(
            (
                start_node,
                float(position),
                _line_part(line, 0.0, position, reverse=True),
                (end_node, start_node),
            )
        )
    if info["direction"] in {"B", "F"}:
        options.append(
            (
                end_node,
                float(line.length - position),
                _line_part(line, position, line.length),
                (start_node, end_node),
            )
        )
    return options


def _target_options(
    info: dict[str, Any], position: float
) -> list[
    tuple[
        tuple[float, float],
        float,
        shapely_geometry.LineString | None,
        tuple[tuple[float, float], tuple[float, float]],
    ]
]:
    """Legal moves from either segment endpoint to a snapped D."""
    line = info["line"]
    start_node = _node_key(line.coords[0])
    end_node = _node_key(line.coords[-1])
    options = []
    if info["direction"] in {"B", "F"}:
        options.append(
            (
                start_node,
                float(position),
                _line_part(line, 0.0, position),
                (start_node, end_node),
            )
        )
    if info["direction"] in {"B", "T"}:
        options.append(
            (
                end_node,
                float(line.length - position),
                _line_part(line, position, line.length, reverse=True),
                (end_node, start_node),
            )
        )
    return options


def _same_road_candidate(
    info: dict[str, Any], start_position: float, end_position: float
) -> tuple[float, shapely_geometry.LineString] | None:
    """Return a direct same-segment path only when its direction is legal."""
    if (
        end_position > start_position + 1e-9
        and info["direction"] in {"B", "F"}
    ):
        geometry = _line_part(info["line"], start_position, end_position)
        return float(end_position - start_position), geometry
    if (
        start_position > end_position + 1e-9
        and info["direction"] in {"B", "T"}
    ):
        geometry = _line_part(
            info["line"], end_position, start_position, reverse=True
        )
        return float(start_position - end_position), geometry
    return None




In [ ]:
def _merge_ordered(parts: list[shapely_geometry.LineString | None]):
    valid = [
        part
        for part in parts
        if part is not None and not part.is_empty and float(part.length) > 0
    ]
    if not valid:
        return None
    if len(valid) == 1:
        return valid[0]
    return shapely_ops.linemerge(shapely_geometry.MultiLineString(valid))


def _route_rng(route_id: str, global_seed: int) -> random.Random:
    """Derive a stable per-route RNG; independent of row and process order."""
    digest = hashlib.sha256(f"{global_seed}|{route_id}".encode("utf-8")).digest()
    return random.Random(int.from_bytes(digest[:16], byteorder="big", signed=False))




**路径集合：** 先求严格最短路，再尝试一条不超过其 120% 的替代路。


In [ ]:
def enumerate_one_path_set(
    graph: nx.DiGraph,
    source_info: dict[str, Any],
    source_position: float,
    target_info: dict[str, Any],
    target_position: float,
    *,
    max_ratio: float = 1.20,
    max_attempts: int = 5,
    rng: random.Random,
) -> list[dict[str, Any]]:
    """Return the strict shortest path and at most one legal <=120% alternative."""
    if max_ratio < 1.0:
        raise ValueError("max_ratio 必须不小于 1")
    if max_attempts < 1:
        raise ValueError("max_attempts 必须为正整数")

    source_options = _source_options(source_info, source_position)
    target_options = _target_options(target_info, target_position)

    def best_path(banned_edge: tuple | None = None) -> dict[str, Any] | None:
        best = None
        if source_info["road_idx"] == target_info["road_idx"]:
            direct = _same_road_candidate(
                source_info, source_position, target_position
            )
            direct_edge = (
                (
                    _node_key(source_info["line"].coords[0]),
                    _node_key(source_info["line"].coords[-1]),
                )
                if target_position > source_position
                else (
                    _node_key(source_info["line"].coords[-1]),
                    _node_key(source_info["line"].coords[0]),
                )
            )
            if direct is not None and (
                banned_edge is None or direct_edge != banned_edge
            ):
                geometry = _merge_ordered([direct[1]])
                if geometry is not None and not geometry.is_empty:
                    best = {
                        "dist_m": float(direct[0]),
                        "n_edges": 1,
                        "geometry": geometry,
                        "edge_sig": (
                            (
                                "direct",
                                source_info["road_idx"],
                                round(source_position, 6),
                                round(target_position, 6),
                            ),
                        ),
                        "used_edges": [direct_edge],
                    }

        route_graph = (
            graph
            if banned_edge is None
            else nx.subgraph_view(
                graph, filter_edge=lambda u, v: (u, v) != banned_edge
            )
        )
        for source_node, source_cost, source_geom, source_edge in source_options:
            if banned_edge is not None and source_edge == banned_edge:
                continue
            for target_node, target_cost, target_geom, target_edge in target_options:
                if banned_edge is not None and target_edge == banned_edge:
                    continue
                try:
                    core_cost, node_path = nx.single_source_dijkstra(
                        route_graph,
                        source_node,
                        target_node,
                        weight="length",
                    )
                except (nx.NetworkXNoPath, nx.NodeNotFound):
                    continue
                core_edges = list(zip(node_path[:-1], node_path[1:]))
                distance = float(source_cost) + float(core_cost) + float(target_cost)
                if best is not None and distance >= best["dist_m"] - 1e-9:
                    continue
                core_parts = [
                    graph.edges[u, v]["geometry"] for u, v in core_edges
                ]
                geometry = _merge_ordered(
                    [source_geom, *core_parts, target_geom]
                )
                if geometry is None or geometry.is_empty:
                    continue
                best = {
                    "dist_m": distance,
                    "n_edges": (
                        len(core_edges)
                        + int(source_geom is not None)
                        + int(target_geom is not None)
                    ),
                    "geometry": geometry,
                    "edge_sig": (
                        ("source", source_node),
                        *core_edges,
                        ("target", target_node),
                    ),
                    "used_edges": list(
                        dict.fromkeys([source_edge, *core_edges, target_edge])
                    ),
                }
        return best

    shortest = best_path()
    if shortest is None or shortest["dist_m"] <= 0:
        return []
    shortest["ratio"] = 1.0
    selected = [shortest]

    edges_to_try = list(dict.fromkeys(shortest["used_edges"]))
    rng.shuffle(edges_to_try)
    for banned_edge in edges_to_try[:max_attempts]:
        alternative = best_path(banned_edge)
        if alternative is None:
            continue
        if alternative["edge_sig"] == shortest["edge_sig"]:
            continue
        if alternative["dist_m"] > shortest["dist_m"] * max_ratio + 1e-7:
            continue
        alternative["ratio"] = float(
            alternative["dist_m"] / shortest["dist_m"]
        )
        selected.append(alternative)
        break

    for candidate in selected:
        candidate.pop("used_edges", None)
        candidate.pop("edge_sig", None)
    return selected




In [ ]:
def _empty_paths(projected_crs: str) -> gpd.GeoDataFrame:
    return gpd.GeoDataFrame(
        columns=[
            "route_id",
            "taxi_id",
            "trip_date",
            "trip_id",
            "start_time",
            "end_time",
            "alt_rank",
            "dist_m",
            "ratio",
            "is_best",
            "n_edges",
            "snap_s_m",
            "snap_e_m",
            "status",
            "geometry",
        ],
        geometry="geometry",
        crs=projected_crs,
    )




In [ ]:
def generate_path_sets(
    trips: pd.DataFrame,
    routing_roads: gpd.GeoDataFrame,
    graph: nx.DiGraph,
    *,
    projected_crs: str = "EPSG:32650",
    max_snap_m: float = 500.0,
    max_ratio: float = 1.20,
    max_attempts: int = 5,
    random_seed: int = 42,
) -> tuple[gpd.GeoDataFrame, pd.DataFrame]:
    """Generate one strict shortest and at most one reproducible near-optimal path."""
    work = trips.reset_index(drop=True).copy()
    starts, ends, invalid_positions = make_od_points(
        work, projected_crs=projected_crs
    )
    start_snap = snap_to_roads(
        starts, routing_roads, max_snap_m=max_snap_m
    )
    end_snap = snap_to_roads(ends, routing_roads, max_snap_m=max_snap_m)
    roads_by_idx = routing_roads.set_index("road_idx")

    path_records: list[dict[str, Any]] = []
    status_records: list[dict[str, Any]] = []
    for position, trip in work.iterrows():
        route_id = str(trip["route_id"])
        common = {"route_id": route_id}
        for column in ("taxi_id", "trip_date", "trip_id", "start_time", "end_time"):
            if column in work.columns:
                common[column] = trip[column]

        status = "ok"
        alt_status = "not_applicable"
        message = ""
        minimum = np.nan
        candidates: list[dict[str, Any]] = []
        snap_s_m = np.nan
        snap_e_m = np.nan

        if position in invalid_positions:
            status, message = "invalid_od", "起终点坐标缺失、非有限或越界"
        else:
            ss = start_snap.loc[position]
            es = end_snap.loc[position]
            if bool(ss["matched"]):
                snap_s_m = float(ss["snap_m"])
            if bool(es["matched"]):
                snap_e_m = float(es["snap_m"])
            if not bool(ss["matched"]) or not bool(es["matched"]):
                status, message = "snap_far", "起点或终点距详细路网超过吸附上限"
            elif (
                float(trip["start_lon"]) == float(trip["end_lon"])
                and float(trip["start_lat"]) == float(trip["end_lat"])
            ):
                status, message, minimum = (
                    "zero_distance",
                    "起点与终点完全相同",
                    0.0,
                )
            elif ss["point"].distance(es["point"]) <= 1e-6:
                status, message, minimum = (
                    "empty_geom",
                    "起终点吸附到同一位置，不能形成正长度路径",
                    0.0,
                )
            else:
                source_info = _road_info(roads_by_idx, int(ss["road_idx"]))
                target_info = _road_info(roads_by_idx, int(es["road_idx"]))
                candidates = enumerate_one_path_set(
                    graph,
                    source_info,
                    float(ss["position"]),
                    target_info,
                    float(es["position"]),
                    max_ratio=max_ratio,
                    max_attempts=max_attempts,
                    rng=_route_rng(route_id, random_seed),
                )
                if not candidates:
                    status, message = "no_path", "路网中没有合法路径"

        if candidates:
            minimum = float(candidates[0]["dist_m"])
            alt_status = "ok" if len(candidates) == 2 else "no_valid_alternative"
            if len(candidates) == 1:
                message = (
                    f"最多禁用最短路边 {max_attempts} 次后，"
                    f"未找到长度不超过最短路 {max_ratio:.0%} 的不同路径"
                )
            for rank, candidate in enumerate(candidates, start=1):
                path_records.append(
                    {
                        **common,
                        "alt_rank": rank,
                        "dist_m": float(candidate["dist_m"]),
                        "ratio": float(candidate["ratio"]),
                        "is_best": rank == 1,
                        "n_edges": int(candidate["n_edges"]),
                        "snap_s_m": snap_s_m,
                        "snap_e_m": snap_e_m,
                        "status": "ok",
                        "geometry": candidate["geometry"],
                    }
                )

        status_records.append(
            {
                **common,
                "status": status,
                "message": message,
                "min_dist_m": minimum,
                "path_count": len(candidates),
                "alt_status": alt_status,
                "snap_s_m": snap_s_m,
                "snap_e_m": snap_e_m,
                "max_snap_m": float(max_snap_m),
                "max_ratio": float(max_ratio),
                "max_attempts": int(max_attempts),
                "random_seed": int(random_seed),
            }
        )

    paths_gdf = (
        gpd.GeoDataFrame(
            path_records, geometry="geometry", crs=projected_crs
        )
        if path_records
        else _empty_paths(projected_crs)
    )
    status_df = pd.DataFrame(
        status_records,
        columns=[
            "route_id",
            "taxi_id",
            "trip_date",
            "trip_id",
            "start_time",
            "end_time",
            "status",
            "message",
            "min_dist_m",
            "path_count",
            "alt_status",
            "snap_s_m",
            "snap_e_m",
            "max_snap_m",
            "max_ratio",
            "max_attempts",
            "random_seed",
        ],
    )
    return paths_gdf, status_df





In [ ]:
def solve_path_sets(
    trips: Any,
    detailed_road_path: Path,
    config: PipelineConfig,
) -> tuple[Any, Any]:
    require_modules(
        "numpy", "pandas", "geopandas", "shapely", "networkx"
    )
    detailed_road_path = Path(detailed_road_path)
    if not detailed_road_path.exists():
        raise FileNotFoundError(f"找不到详细路由路网：{detailed_road_path}")
    raw_roads = gpd.read_file(detailed_road_path)
    segments = validate_detailed_roads(
        raw_roads,
        projected_crs=config.projected_crs,
    )
    retained_segments, graph = prepare_routing_network(segments)
    return generate_path_sets(
        trips,
        retained_segments,
        graph,
        projected_crs=config.projected_crs,
        max_snap_m=config.routing_snap_tolerance_m,
        max_ratio=config.alternative_max_ratio,
        max_attempts=config.alternative_max_attempts,
        random_seed=config.random_seed,
    )


def validate_path_sets(
    paths: Any,
    *,
    max_ratio: float = 1.20,
) -> None:
    require_modules("numpy", "pandas", "geopandas")
    require_crs(paths, "路径集")
    if paths.empty:
        return
    require_columns(
        paths,
        ["route_id", "alt_rank", "dist_m", "ratio", "is_best", "geometry"],
        "路径集",
    )
    for route_id, group in paths.groupby("route_id", sort=False):
        ordered = group.sort_values("alt_rank")
        ranks = ordered["alt_rank"].astype(int).tolist()
        if ranks not in ([1], [1, 2]):
            raise ValueError(f"{route_id} 的路径 rank 必须为 [1] 或 [1,2]")
        distances = pd.to_numeric(ordered["dist_m"], errors="coerce").to_numpy()
        ratios = pd.to_numeric(ordered["ratio"], errors="coerce").to_numpy()
        if not np.isfinite(distances).all() or (distances <= 0).any():
            raise ValueError(f"{route_id} 存在非正或无效路径长度")
        if np.any(np.diff(distances) < -1e-7):
            raise ValueError(f"{route_id} 的路径没有按长度升序")
        if not math.isclose(float(ratios[0]), 1.0, rel_tol=0, abs_tol=1e-7):
            raise ValueError(f"{route_id} 的严格最短路 ratio 不为 1")
        if (ratios > max_ratio + 1e-7).any():
            raise ValueError(f"{route_id} 存在超过 {max_ratio:.2f} 的替代路")
        if ordered.geometry.isna().any() or ordered.geometry.is_empty.any():
            raise ValueError(f"{route_id} 存在空路径几何")


def export_path_outputs(
    paths: Any,
    status: Any,
    output_gpkg: Path,
    output_status_csv: Path,
    config: PipelineConfig,
) -> None:
    validate_path_sets(paths, max_ratio=config.alternative_max_ratio)
    write_gpkg_layer(paths, output_gpkg, layer="path_sets")
    output_status_csv = Path(output_status_csv)
    if output_status_csv.exists():
        raise FileExistsError(f"输出已存在：{output_status_csv}")
    output_status_csv.parent.mkdir(parents=True, exist_ok=True)
    status.to_csv(output_status_csv, index=False, encoding="utf-8-sig")


## 7. 轨迹线与最优路径求交

筛选早高峰研究区行程，并比较实际轨迹与最短路径长度。


In [ ]:
from __future__ import annotations

def to_local_datetimes(
    values: Any,
    *,
    timezone_name: str,
) -> Any:
    require_modules("pandas")
    series = pd.Series(values, index=getattr(values, "index", None))
    nonempty = series.notna()
    if isinstance(series.dtype, pd.DatetimeTZDtype):
        return pd.to_datetime(
            series, errors="coerce"
        ).dt.tz_convert(timezone_name)
    if pd.api.types.is_datetime64_any_dtype(series.dtype):
        return pd.to_datetime(
            series, errors="coerce"
        ).dt.tz_localize(
            timezone_name, ambiguous="NaT", nonexistent="NaT"
        )

    text_values = series.loc[nonempty].astype(str).str.strip()
    numeric_strings = bool(
        text_values.str.fullmatch(
            r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
        ).all()
    )
    if pd.api.types.is_numeric_dtype(series.dtype) or numeric_strings:
        numeric = pd.to_numeric(series, errors="coerce")
        return pd.to_datetime(
            numeric, unit="s", utc=True, errors="coerce"
        ).dt.tz_convert(timezone_name)

    parsed = pd.to_datetime(series, errors="coerce")
    if isinstance(parsed.dtype, pd.DatetimeTZDtype):
        return parsed.dt.tz_convert(timezone_name)
    return parsed.dt.tz_localize(
        timezone_name, ambiguous="NaT", nonexistent="NaT"
    )


def _hhmm_to_time(text: str) -> clock_time:
    hour, minute = (int(part) for part in text.split(":"))
    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        raise ValueError(f"无效时间：{text}")
    return clock_time(hour, minute)


def _overlaps_daily_window(
    start: Any,
    end: Any,
    *,
    window_start: clock_time,
    window_end: clock_time,
) -> bool:
    if pd.isna(start) or pd.isna(end) or end <= start:
        return False
    current_day = start.normalize()
    final_day = end.normalize()
    while current_day <= final_day:
        window_start_at = current_day + pd.Timedelta(
            hours=window_start.hour, minutes=window_start.minute
        )
        window_end_at = current_day + pd.Timedelta(
            hours=window_end.hour, minutes=window_end.minute
        )
        if window_end_at <= window_start_at:
            window_end_at += pd.Timedelta(days=1)
        if start < window_end_at and end > window_start_at:
            return True
        current_day += pd.Timedelta(days=1)
    return False




In [ ]:
def flag_morning_peak(
    frame: Any,
    config: PipelineConfig,
    *,
    start_col: str = "start_time",
    end_col: str = "end_time",
) -> Any:
    require_modules("pandas", "numpy")
    require_columns(frame, [start_col, end_col], "早高峰判断输入")
    result = frame.copy()
    start_local = to_local_datetimes(
        result[start_col], timezone_name=config.timezone_name
    )
    end_local = to_local_datetimes(
        result[end_col], timezone_name=config.timezone_name
    )
    valid_time = start_local.notna() & end_local.notna() & end_local.gt(start_local)
    morning_start = _hhmm_to_time(config.morning_start)
    morning_end = _hhmm_to_time(config.morning_end)
    result["morning_peak"] = [
        _overlaps_daily_window(
            start,
            end,
            window_start=morning_start,
            window_end=morning_end,
        )
        for start, end in zip(start_local, end_local)
    ]
    result["time_status"] = np.where(valid_time, "ok", "invalid_time")
    result["start_time_beijing"] = start_local.astype("string")
    result["end_time_beijing"] = end_local.astype("string")
    return result


def _coerce_area_frame(
    study_area: Any,
    config: PipelineConfig,
) -> Any:
    require_modules("geopandas")
    if isinstance(study_area, (str, Path)):
        area_frame = gpd.read_file(study_area)
    elif hasattr(study_area, "geometry"):
        area_frame = study_area.copy()
    else:
        area_frame = gpd.GeoDataFrame(
            {"area_id": [1]},
            geometry=[study_area],
            crs=config.geographic_crs,
        )
    require_crs(area_frame, "研究区")
    area_frame = area_frame.loc[
        area_frame.geometry.notna() & ~area_frame.geometry.is_empty
    ].copy()
    if area_frame.empty:
        raise ValueError("研究区没有有效几何")
    return area_frame


def flag_study_area(
    paths: Any,
    study_area: Any,
    config: PipelineConfig,
) -> Any:
    require_modules("geopandas", "shapely")
    require_crs(paths, "待标记轨迹")
    area_frame = _coerce_area_frame(study_area, config).to_crs(paths.crs)
    area_union = (
        area_frame.geometry.union_all()
        if hasattr(area_frame.geometry, "union_all")
        else area_frame.unary_union
    )
    result = paths.copy()
    valid = result.geometry.notna() & ~result.geometry.is_empty
    result["in_study_area"] = False
    result.loc[valid, "in_study_area"] = result.loc[
        valid, "geometry"
    ].intersects(area_union)
    return result


def select_morning_study_area_paths(
    path_sets: Any,
    study_area: Any,
    config: PipelineConfig,
) -> tuple[Any, Any]:
    marked = flag_morning_peak(path_sets, config)
    marked = flag_study_area(marked, study_area, config)
    morning_marked = marked.loc[
        marked["morning_peak"] & marked["time_status"].eq("ok")
    ].copy()
    morning_inside = morning_marked.loc[
        morning_marked["in_study_area"]
    ].copy()
    return morning_marked, morning_inside




In [ ]:
def _valid_line_geometry(geometry: Any) -> bool:
    return bool(
        geometry is not None
        and not geometry.is_empty
        and geometry.is_valid
        and geometry.geom_type in {"LineString", "MultiLineString"}
    )




**绕行判定：** 实际长度与严格最短路长度之比达到 1.5 即为绕行。


In [ ]:
def label_detours(
    actual_lines: Any,
    path_sets: Any,
    config: PipelineConfig,
) -> Any:
    require_modules("numpy", "pandas", "geopandas")
    require_crs(actual_lines, "实际轨迹")
    require_crs(path_sets, "最优路径集")
    require_columns(actual_lines, ["route_id", "geometry"], "实际轨迹")
    require_columns(
        path_sets,
        ["route_id", "alt_rank", "dist_m", "geometry"],
        "最优路径集",
    )
    if actual_lines["route_id"].astype(str).duplicated().any():
        raise ValueError("实际轨迹 route_id 必须唯一")
    shortest = path_sets.loc[
        pd.to_numeric(path_sets["alt_rank"], errors="coerce").eq(1)
    ].copy()
    if shortest["route_id"].astype(str).duplicated().any():
        raise ValueError("每个 route_id 必须恰好对应至多一条 alt_rank=1 最短路")

    actual_projected = actual_lines.to_crs(config.projected_crs).copy()
    shortest_projected = shortest.to_crs(config.projected_crs).copy()
    shortest_lookup = shortest_projected.set_index(
        shortest_projected["route_id"].astype(str), drop=False
    )
    rows: list[dict[str, Any]] = []
    for _, actual in actual_projected.iterrows():
        route_id = str(actual["route_id"])
        record = actual.to_dict()
        record["actual_length_m"] = float("nan")
        record["optimal_length_m"] = float("nan")
        record["detour_ratio"] = float("nan")
        record["detour"] = pd.NA
        record["comparison_status"] = "unmatched_route"

        if (
            actual.geometry is None
            or actual.geometry.is_empty
            or actual.geometry.geom_type not in {"LineString", "MultiLineString"}
        ):
            record["comparison_status"] = "invalid_geometry"
            rows.append(record)
            continue
        actual_length = float(actual.geometry.length)
        record["actual_length_m"] = actual_length
        if not math.isfinite(actual_length):
            record["comparison_status"] = "invalid_geometry"
            rows.append(record)
            continue
        if actual_length <= 0:
            record["comparison_status"] = "zero_actual_length"
            rows.append(record)
            continue
        if not actual.geometry.is_valid:
            record["comparison_status"] = "invalid_geometry"
            rows.append(record)
            continue
        if route_id not in shortest_lookup.index:
            rows.append(record)
            continue

        optimal = shortest_lookup.loc[route_id]
        if isinstance(optimal, pd.DataFrame):
            raise ValueError(f"{route_id} 对应多条严格最短路")
        optimal_length = float(
            pd.to_numeric(
                pd.Series([optimal["dist_m"]]), errors="coerce"
            ).iloc[0]
        )
        record["optimal_length_m"] = optimal_length
        if not math.isfinite(optimal_length) or optimal_length <= 0:
            record["comparison_status"] = "nonpositive_optimal_length"
            rows.append(record)
            continue
        if not _valid_line_geometry(optimal.geometry):
            record["comparison_status"] = "invalid_optimal_geometry"
            rows.append(record)
            continue

        ratio = actual_length / optimal_length
        record["detour_ratio"] = ratio
        record["detour"] = int(ratio >= config.detour_ratio_threshold)
        record["comparison_status"] = "ok"
        rows.append(record)

    if rows:
        result = gpd.GeoDataFrame(
            rows, geometry="geometry", crs=config.projected_crs
        )
    else:
        result = actual_projected.iloc[0:0].copy()
        result["actual_length_m"] = pd.Series(dtype="float64")
        result["optimal_length_m"] = pd.Series(dtype="float64")
        result["detour_ratio"] = pd.Series(dtype="float64")
        result["comparison_status"] = pd.Series(dtype="object")
        result["detour"] = pd.Series(dtype="Int64")
    result["detour"] = pd.array(result["detour"], dtype="Int64")
    return result




In [ ]:
def detour_summary(detour_frame: Any) -> dict[str, Any]:
    require_modules("pandas")
    valid = detour_frame.loc[
        detour_frame["comparison_status"].eq("ok")
        & detour_frame["detour"].notna()
    ]
    denominator = len(valid)
    numerator = int(valid["detour"].sum()) if denominator else 0
    return {
        "eligible_route_count": denominator,
        "detour_route_count": numerator,
        "detour_rate": numerator / denominator if denominator else None,
        "excluded_by_status": detour_frame.loc[
            ~detour_frame["comparison_status"].eq("ok"),
            "comparison_status",
        ].value_counts(dropna=False).to_dict(),
    }


def export_final_analysis(
    morning_marked: Any,
    morning_inside: Any,
    detours: Any,
    output_dir: Path,
) -> dict[str, Path]:
    require_modules("pandas", "geopandas")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = {
        "morning_marked": output_dir / "morning_paths_with_area_flag.gpkg",
        "morning_inside": output_dir / "morning_paths_in_study_area.gpkg",
        "detours_gpkg": output_dir / "detour_results.gpkg",
        "detours_csv": output_dir / "detour_results.csv",
        "summary": output_dir / "detour_summary.json",
    }
    for path in paths.values():
        if path.exists():
            raise FileExistsError(f"输出已存在：{path}")
    write_gpkg_layer(
        morning_marked, paths["morning_marked"], layer="morning_paths"
    )
    write_gpkg_layer(
        morning_inside, paths["morning_inside"], layer="morning_inside"
    )
    write_gpkg_layer(detours, paths["detours_gpkg"], layer="detours")
    pd.DataFrame(detours.drop(columns="geometry")).to_csv(
        paths["detours_csv"], index=False, encoding="utf-8-sig"
    )
    paths["summary"].write_text(
        json.dumps(detour_summary(detours), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return paths


## 8. ArcGIS 可选步骤

用于候选区融合、最终研究区制作和时空立方体；普通 Python 流程不依赖本节。


**人工研究区：** 候选区在 ArcGIS Pro 中提取、融合并框选为最终面要素。


In [ ]:
from __future__ import annotations

def _load_licensed_arcpy(stage: str) -> tuple[Any | None, StageResult | None]:
    try:
        arcpy = importlib.import_module("arcpy")
    except Exception as exc:
        return None, stage_skipped(
            stage, f"未检测到 arcpy：{type(exc).__name__}: {exc}"
        )
    try:
        product_status = arcpy.CheckProduct("ArcInfo")
    except Exception as exc:
        return None, stage_skipped(
            stage, f"无法检查 ArcGIS Pro 许可：{type(exc).__name__}: {exc}"
        )
    if str(product_status) not in {"Available", "AlreadyInitialized"}:
        return None, stage_skipped(
            stage, f"ArcGIS Pro 许可不可用：{product_status}"
        )
    return arcpy, None


def arcgis_finalize_study_area(
    candidate_features: Path | str,
    output_feature_class: Path | str,
    *,
    where_clause: str | None = None,
    manual_clip_features: Path | str | None = None,
) -> StageResult:
    stage = "8_arcgis_finalize_study_area"
    started = time.perf_counter()
    arcpy, skipped = _load_licensed_arcpy(stage)
    if skipped is not None:
        return skipped
    candidate_features = Path(candidate_features)
    output_feature_class = Path(output_feature_class)
    inputs = [candidate_features]
    if manual_clip_features is not None:
        inputs.append(Path(manual_clip_features))
    missing = [path for path in inputs if not arcpy.Exists(str(path))]
    if missing:
        return stage_skipped(
            stage, f"ArcGIS 人工处理输入缺失：{missing}", inputs=inputs
        )
    if arcpy.Exists(str(output_feature_class)):
        return stage_skipped(
            stage,
            f"为避免覆盖，输出已存在：{output_feature_class}",
            inputs=inputs,
            outputs=[output_feature_class],
        )
    try:
        candidate_layer = "taxi_candidate_layer"
        arcpy.management.MakeFeatureLayer(
            str(candidate_features), candidate_layer
        )
        if where_clause:
            arcpy.management.SelectLayerByAttribute(
                candidate_layer, "NEW_SELECTION", where_clause
            )
        source_for_dissolve = candidate_layer
        temporary_clip = None
        if manual_clip_features is not None:
            temporary_clip = "memory/taxi_candidate_manual_clip"
            arcpy.analysis.PairwiseClip(
                candidate_layer,
                str(manual_clip_features),
                temporary_clip,
            )
            source_for_dissolve = temporary_clip
        arcpy.management.Dissolve(
            source_for_dissolve,
            str(output_feature_class),
            multi_part="MULTI_PART",
        )
        if temporary_clip and arcpy.Exists(temporary_clip):
            arcpy.management.Delete(temporary_clip)
        return StageResult(
            stage=stage,
            status=StageStatus.SUCCESS,
            message="候选区选择、可选人工框选裁剪及融合完成",
            inputs=[str(path) for path in inputs],
            outputs=[str(output_feature_class)],
            elapsed_sec=time.perf_counter() - started,
        )
    except Exception as exc:
        return stage_failed(
            stage,
            "ArcGIS 候选区人工处理失败",
            exc,
            inputs=inputs,
            outputs=[output_feature_class],
            elapsed_sec=time.perf_counter() - started,
        )




In [ ]:
def arcgis_create_space_time_cube(
    input_features: Path | str,
    output_cube: Path | str,
    time_field: str,
    *,
    tool_parameters: Mapping[str, Any] | None = None,
) -> StageResult:
    stage = "8_arcgis_space_time_cube"
    started = time.perf_counter()
    arcpy, skipped = _load_licensed_arcpy(stage)
    if skipped is not None:
        return skipped
    input_features = Path(input_features)
    output_cube = Path(output_cube)
    if not arcpy.Exists(str(input_features)):
        return stage_skipped(
            stage,
            f"时空立方体输入缺失：{input_features}",
            inputs=[input_features],
        )
    if arcpy.Exists(str(output_cube)):
        return stage_skipped(
            stage,
            f"为避免覆盖，输出已存在：{output_cube}",
            inputs=[input_features],
            outputs=[output_cube],
        )
    stpm = getattr(arcpy, "stpm", None)
    tool = getattr(stpm, "CreateSpaceTimeCube", None)
    if not callable(tool):
        return stage_skipped(
            stage,
            "当前 ArcGIS 安装没有 stpm.CreateSpaceTimeCube 工具",
            inputs=[input_features],
        )
    parameters = {
        "in_features": str(input_features),
        "output_cube": str(output_cube),
        "time_field": time_field,
    }
    parameters.update(dict(tool_parameters or {}))
    try:
        tool(**parameters)
        return StageResult(
            stage=stage,
            status=StageStatus.SUCCESS,
            message="ArcGIS 时空立方体创建完成",
            inputs=[str(input_features)],
            outputs=[str(output_cube)],
            elapsed_sec=time.perf_counter() - started,
        )
    except Exception as exc:
        return stage_failed(
            stage,
            "ArcGIS 时空立方体创建失败；请核对当前 Pro 版本的工具参数",
            exc,
            inputs=[input_features],
            outputs=[output_cube],
            elapsed_sec=time.perf_counter() - started,
        )
